# EWMA-Toleranzintervall-Pairtrading über Cliquen (Lin, Chen & Syu 2021)

**Was dieses Notebook tut:** simuliert die Pairtrading-Strategie aus Lin, Chen & Syu (2021,
"Multi-asset pair-trading strategy: A statistical learning approach") auf den Mitgliedern der
in `02_tda_clustering` gefundenen und in `data/processed/cliques/` gecachten Cliquen, und
optimiert die Signal-Parameter dieser Strategie automatisiert mit Optuna.

**Methodischer Unterschied zu einer klassischen RSI/Bollinger-Engine:** statt einzelne Assets
gegen ihre eigenen Bänder zu handeln, wird hier für JEDES Paar innerhalb einer Clique die
Log-Return-Differenz `Rt = r_Ai,t - r_Aj,t` gebildet, per EWMA-Volatilität normiert
(`Xt = Rt / sigma_t`), und über ein nichtparametrisches Wilks-Toleranzintervall (statt eines
willkürlichen Z-Score-Schwellenwerts) auf Signifikanz geprüft — datengetrieben statt
willkürlich, siehe Additional-Info-PDF.

**Wichtige Abgrenzung** (nicht mit dem Preis-Level-Spread aus der ADF/KPSS/Hurst-
Stationaritätsdiagnostik in `03_signal_generation/TradeData.ipynb` verwechseln): dort wurde die
Renditedifferenz durch einen Preis-Level-Spread ersetzt, weil DAS der richtige Prozess für
klassische Stationaritätstests ist. HIER, für die Lin-et-al.-Signalgenerierung, ist die
Renditedifferenz selbst die vom Paper vorgeschriebene, korrekte Größe — sie bleibt unverändert.

**Aufbau dieses Notebooks (bewusst schlank gehalten — EIN Tuning-Weg, kein Nebeneinander
mehrerer Varianten):**
1. Daten & Cliquen-Cache laden.
2. Kurze, formelfreie Erklärung, wie die Handelssignale überhaupt entstehen (Abschnitt 4) —
   Grundlage, um die Parameter im Optuna-Suchraum (Abschnitt 5) inhaltlich einordnen zu können.
3. Optuna-Tuning (Abschnitt 5): Hyperparameter-Suche
   (voller Walk-Forward-Backtest je Trial) mit persistentem SQLite-Storage und Live-Dashboard.
4. Einmalige finale Auswertung auf einem während des gesamten Tunings unberührten Testzeitraum
   (Abschnitt 5b) — DAS ist die Zahl, die als Out-of-Sample-Ergebnis zitiert werden sollte.

Die eigentliche Logik (EWMA, Spreads, Wilks-Toleranzintervall, Signal-/Trade-Simulation,
Equity-Kurve, Kennzahlen, Plots, Walk-Forward-Fenster) steckt komplett in
`pairtrading_engine.py`; die Tuning-Logik (Optuna, Pruning, Zielgröße,
persistenter Storage) in `tune_backtest.py`. Dieses Notebook ist nur das Interface darüber —
es enthält selbst keine Backtest- oder Optimierungslogik. Die Kern-Funktion
`wilks_tolerance_limits` ist in `test_pairtrading_engine.py` gegen die zugrundeliegende
Beta-Verteilungsformel abgesichert (nicht gegen einen unsicheren Tabellenwert) — vor jeder
produktiven Nutzung sollten diese Tests grün sein (`python test_pairtrading_engine.py`).


In [1]:
# =========================
# 1. Setup & Imports
# =========================
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

import pairtrading_engine as pte
import tune_backtest as tb
from Src.config import (
    ASSET_DAILY_PATH,
    ASSET_ID_COL,
    ASSETS_PATH,
    CLOSE_COL,
    MARKET_SYMBOL_COL,
    OHLCV_DIR,
    PROCESSED_DIR,
    PROJECT_ROOT,
    TIMESTAMP_COL,
)
from Src.clustering.helpers import STABLECOIN_SYMBOLS
from Src.signals.spread import compute_price_spread
from Src.signals.bollinger import build_pairwise_price_spreads, compute_exit_mean
from Src.signals.garch_tolerance import compute_garch_adjusted_spreads
from Src.signals.tolerance import (
    build_pairwise_return_spreads,
    compute_adjusted_spreads,
    compute_max_adjusted_spread,
    generate_tolerance_signals,
)

print("Imports OK")

Imports OK


# 1. Daten laden

Lädt die 5-Minuten-Kerzen-OHLCV-Daten aus `data/processed/ohlcv/` und mappt sie über die
Asset-Stammdaten (`assets.parquet`) auf `asset_id` — dieselbe ID, mit der auch die Cliquen im
nächsten Abschnitt referenziert werden. Stablecoins werden ausgeschlossen (`STABLECOIN_SYMBOLS`
aus `helperfunctions.py`), da Pairtrading zwischen zwei Stablecoins (beide ~konstant bei 1 USD)
kein sinnvolles Signal liefern kann — die Renditedifferenz wäre strukturell nahe Null,
unabhängig von jedem Marktereignis.


In [2]:
assets_df = pd.read_parquet(ASSETS_PATH)[["asset_id", "symbol"]].drop_duplicates()
id_to_symbol = dict(zip(assets_df["asset_id"], assets_df["symbol"]))

excluded_ids_stablecoins: set[int] = set(
    assets_df.loc[assets_df["symbol"].isin(STABLECOIN_SYMBOLS), "asset_id"].tolist()
)

ohlcv_files = sorted(OHLCV_DIR.glob("*.parquet"))
raw = pd.concat([pd.read_parquet(f) for f in ohlcv_files], ignore_index=True)
raw["base_symbol"] = raw[MARKET_SYMBOL_COL].str.replace("USDT$", "", regex=True)
raw = raw.merge(assets_df.rename(columns={"symbol": "base_symbol"}), on="base_symbol", how="left")
raw = raw.dropna(subset=["asset_id"])
raw["asset_id"] = raw["asset_id"].astype(int)
raw[TIMESTAMP_COL] = pd.to_datetime(raw[TIMESTAMP_COL], utc=True)
raw = raw.loc[~raw["asset_id"].isin(excluded_ids_stablecoins)]
ohlcv_df = raw[[TIMESTAMP_COL, ASSET_ID_COL, CLOSE_COL]].copy()

data_min = ohlcv_df[TIMESTAMP_COL].min().date()
data_max = ohlcv_df[TIMESTAMP_COL].max().date()

print(f"Zeitraum: {data_min} → {data_max}")
print(f"OHLCV: {ohlcv_df.shape[0]:,} Zeilen | Assets: {ohlcv_df[ASSET_ID_COL].nunique()}")

Zeitraum: 2024-07-23 → 2026-07-22
OHLCV: 8,553,834 Zeilen | Assets: 45


# 2. Cliquen-Cache laden

Lädt die von der TDA-Pipeline (`02_tda_clustering`) gefundenen Cliquen aus dem Cache in
`data/processed/cliques/` — **keine eigene Clique-Discovery hier**. Eine Clique ist eine Gruppe
von Assets, die an einem bestimmten Tag (`discovery_date`) laut Vietoris-Rips-Persistenzanalyse
eng topologisch zusammenhängen (siehe Abschnitt 4 für die intuitive Erklärung, warum das für
Pairtrading relevant ist). Zwei Varianten stehen zur Verfügung:
- `raw`: Cliquen direkt auf den Rohreturns.
- `residual`: Cliquen auf Residual-Returns (nach Herausrechnen eines Marktfaktors) — reduziert
  das Risiko, dass eine Clique nur den allgemeinen Marktauf-/abschwung abbildet statt einer
  echten paarweisen Kointegrations-/Zusammenhangsstruktur.

Welche Variante fürs Tuning verwendet wird, ist in Abschnitt 5 (Optuna-Widgets) einstellbar.


In [3]:
CLIQUES_DIR = PROCESSED_DIR / "cliques"


def _load_cliques_cache(variant: str) -> pd.DataFrame:
    path = CLIQUES_DIR / f"cliques_{variant}.parquet"
    if not path.exists():
        raise FileNotFoundError(
            f"{path.relative_to(PROJECT_ROOT)} nicht gefunden -- zuerst "
            "02_tda_clustering/05_clique_cache.ipynb laufen lassen."
        )
    df = pd.read_parquet(path)
    df["discovery_date"] = pd.to_datetime(df["discovery_date"], utc=True)
    df["asset_ids"] = df["asset_ids"].apply(list)  # Parquet gibt Listen als np.ndarray zurück
    return df


cliques_by_variant = {
    "raw": _load_cliques_cache("raw"), #raw bedeutet, dass die Cliquen direkt aus den TDA-Ergebnissen stammen, ohne weitere Filterung oder Anpassung.
    "residual": _load_cliques_cache("residual"), #residual bedeutet, dass die Cliquen aus den TDA-Ergebnissen stammen, aber auf Basis der Residualwerte berechnet wurden, um die Auswirkungen von Trends oder saisonalen Mustern zu reduzieren.
}
basket_labels_available = sorted(cliques_by_variant["raw"]["basket"].unique())

for variant, df in cliques_by_variant.items():
    print(f"cliques_{variant}: {len(df)} Cliquen über {df['discovery_date'].nunique()} Tage "
          f"({df['discovery_date'].min().date()} → {df['discovery_date'].max().date()})")

cliques_raw: 1750 Cliquen über 674 Tage (2024-07-29 → 2026-07-22)
cliques_residual: 1188 Cliquen über 596 Tage (2024-07-29 → 2026-07-22)


# 3. Wie entstehen die Trading-Signale? (einfach erklärt)

Bevor es an das Parameter-Tuning geht, kurz und ohne Formeln: **was steckt eigentlich hinter
den Parametern, die Optuna im nächsten Abschnitt optimiert?**

**1. Eine Clique ist eine Gruppe von Coins, die sich normalerweise ähnlich bewegen.**
Diese Gruppen wurden vorher mit Topologischer Datenanalyse (TDA) gefunden — Coins landen
nur dann zusammen in einer Clique, wenn ihre Kursverläufe über die Zeit eng zusammenhängen.

**2. Für jedes Coin-Paar in der Clique wird laufend die Differenz ihrer Renditen berechnet.**
Bewegen sich zwei Coins normalerweise im Gleichschritt, ist diese Differenz meistens nahe
Null. Wird sie plötzlich groß, bedeutet das: die beiden Coins haben sich kurzfristig
"auseinanderbewegt" — einer ist stärker gestiegen oder gefallen als der andere, obwohl sie
sich eigentlich ähnlich verhalten sollten.

**3. "Groß" wird nicht willkürlich festgelegt, sondern aus der jüngeren Vergangenheit gelernt.**
Für jedes Paar schaut sich die Strategie ein Zeitfenster der letzten Kerzen an (`lookback_candles`
— **einer der Optuna-Suchparameter**) und berechnet daraus eine statistische Toleranzgrenze:
einen Bereich, in dem sich die Renditedifferenz normalerweise bewegt. `p_content` (**ebenfalls
tunbar**) legt fest, wie viel Prozent der historischen Werte innerhalb dieses Bereichs liegen
sollen (z. B. 95 %), `confidence` (fest auf 0.95, siehe Abschnitt 5) sagt, wie sicher man sich
bei dieser Schätzung sein will. Vorteil gegenüber einem festen Schwellenwert: die Grenze passt
sich automatisch an, ob ein Coin gerade ruhig oder volatil ist. Ruhige Phasen -> engere Grenze,
turbulente Phasen -> weitere Grenze.

**4. Verlässt die Renditedifferenz eines Paares diese Toleranzgrenze, entsteht ein Signal.**
Das interpretiert die Strategie als: "diese beiden Coins sind kurzfristig auseinandergelaufen,
das sollte sich wieder normalisieren." Der Trade wettet auf genau diese Rückkehr zum
Normalzustand (Mean-Reversion) — man kauft den relativ zurückgebliebenen Coin (Long) und
verkauft gleichzeitig den relativ vorgelaufenen Coin (Short), sodass nur die *Differenz*
zwischen den beiden zählt, nicht die allgemeine Marktrichtung. `trade_mode` (**tunbar**)
entscheidet, ob dabei NUR das Paar mit dem stärksten Ausschlag pro Zeitpunkt gehandelt wird
(`max_only`, repliziert Lin et al. exakt) oder ALLE Paare, die ihre Grenze verletzen
(`all_significant`).

**5. Der Trade wird wieder geschlossen, sobald sich die Differenz normalisiert hat**
(zurück innerhalb der Toleranzgrenze), oder spätestens nach der eingestellten maximalen
Haltedauer (`max_holding_hours`, **tunbar**), damit keine Position unbegrenzt offenbleibt.
Wie oft die Toleranzgrenze neu geschätzt wird (statt sie über lange Zeit fest zu halten),
steuert `recalib_every` (**tunbar**) — und `max_concurrent_positions` (**tunbar**) begrenzt,
wie viele Positionen gleichzeitig offen sein dürfen (Kapitalallokations-/Risikogrenze).

**Warum das Ganze funktionieren könnte:** die Coins in einer Clique wurden gerade *weil*
sie eng zusammenhängen ausgewählt — kurzfristige Abweichungen von diesem normalen
Zusammenhang gelten als vorübergehende "Übertreibung", die sich statistisch häufig wieder
zurückbildet. Ob und wie stark das tatsächlich zutrifft, hängt genau von den sechs Parametern
ab, die Abschnitt 5 mit Optuna durchsucht — deshalb ist die Signal-Erklärung hier direkt die
Grundlage, um die dortigen Suchraum-Grenzen und Ergebnisse einzuordnen.

**Was NICHT hier optimiert wird** (siehe Abschnitt 5 für die ausführliche Begründung): OB eine
Clique/ein Paar überhaupt als handelbar gilt, wird bereits VOR diesem Notebook entschieden —
durch die TDA-Persistenzschwelle (`min_persistence`, Clique-Discovery) und durch statistische
Stationaritätstests (ADF/KPSS/FDR, `03_signal_generation/TradeData.ipynb`). Diese beiden
vorgelagerten Filter sind theoriegeleitet und bewusst NICHT Teil der Optuna-Suche.


# 4. Schnelltest (feste Parameter, kurzer Zeitraum)

Bevor das teure Optuna-Tuning (Abschnitt 5) laeuft: hier mit EINEM festen
Parametersatz und einem KURZEN Zeitraum (wenige Tage) pruefen, ob die
Signalkette ueberhaupt plausible Ein-/Ausstiege produziert -- schnelles
"lohnt sich ein Tuning ueberhaupt"-Feedback, kein Ersatz fuer das eigentliche
Tuning/die Walk-Forward-Auswertung. Zeigt pro Basket/Zeitraum:
- den Xmax-Verlauf mit Wilks-Toleranzgrenzen und Entry-Markern
  (plot_xmax_with_tolerance_bands, wie in Abschnitt 5c),
- die Trade-Liste,
- eine kompakte Kennzahlen-Uebersicht (compute_backtest_stats).


In [4]:
# =====================================================================
# Abschnitt 4: Schnelltest -- Widgets
# =====================================================================
qt_w_variant = widgets.Dropdown(options=["raw", "residual"], value="raw", description="Cliquen-Variante:", style={"description_width": "160px"})
qt_w_basket = widgets.Dropdown(options=basket_labels_available, value=basket_labels_available[0], description="Basket:", style={"description_width": "160px"})
qt_w_start = widgets.DatePicker(description="Start:", style={"description_width": "160px"})
qt_w_days = widgets.IntSlider(value=5, min=1, max=30, description="Dauer (Tage):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))

# Feste Parameter -- bewusst keine Suche, siehe Abschnitt 5 fuer den Suchraum.
# Default ewma_decay=0.94 ist der RiskMetrics-Tagesdaten-Standard (siehe
# Abschnitt 5/tune_backtest.py-Docstring), fuer 5-Min-Kerzen nur ein Startwert.
qt_w_ewma_decay = widgets.Dropdown(options=[0.89, 0.94, 0.97, 0.985], value=0.94, description="ewma_decay:", style={"description_width": "160px"})
qt_w_lookback = widgets.Dropdown(options=[36, 72, 144, 288], value=144, description="lookback_candles:", style={"description_width": "160px"})
qt_w_p_content = widgets.Dropdown(options=[0.90, 0.95], value=0.95, description="p_content:", style={"description_width": "160px"})
qt_w_exit_lookback = widgets.Dropdown(options=[10, 20, 40, 60, 100], value=40, description="exit_lookback_candles:", style={"description_width": "160px"})
qt_w_confidence = widgets.Dropdown(options=[0.90, 0.95, 0.99], value=0.95, description="confidence:", style={"description_width": "160px"})
qt_w_recalib_every = widgets.IntSlider(value=12, min=6, max=48, description="recalib_every:", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
qt_w_max_holding_hours = widgets.FloatSlider(value=12.0, min=4.0, max=48.0, step=0.5, description="max_holding_hours:", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
qt_w_max_concurrent = widgets.IntSlider(value=10, min=3, max=15, description="max_concurrent_positions:", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
qt_w_trade_mode = widgets.Dropdown(options=["max_only", "all_significant"], value="max_only", description="trade_mode:", style={"description_width": "160px"})
qt_w_fee_bps = widgets.FloatText(value=1.5, description="fee_bps:", style={"description_width": "160px"})

_qt_default_start = pd.Timestamp(max(pd.Timestamp(data_min), pd.Timestamp("2025-01-01"))).date()
qt_w_start.value = _qt_default_start

qt_run_button = widgets.Button(description="Schnelltest starten", button_style="primary", icon="bolt")
qt_output = widgets.Output()
qt_last_result = {}


def _qt_row(widget, description):
    desc_html = widgets.HTML(
        f"<span style='color:#888; font-size:0.9em;'>{description}</span>",
        layout=widgets.Layout(margin="6px 0 0 12px"),
    )
    return widgets.HBox([widget, desc_html])


qt_ui = widgets.VBox([
    widgets.HTML("<b>Universum & Zeitraum</b>"),
    _qt_row(qt_w_variant, "Cliquen-Cache-Variante."),
    _qt_row(qt_w_basket, "Welcher TDA-Basket."),
    _qt_row(qt_w_start, "Start des kurzen Testzeitraums."),
    _qt_row(qt_w_days, "Dauer in Tagen -- kurz halten (paar Tage), sonst ist es kein Schnelltest mehr."),
    widgets.HTML("<b>Feste Parameter</b> (keine Suche -- fuer eine grobe Plausibilitaetspruefung)"),
    _qt_row(qt_w_ewma_decay, "EWMA-Decay fuer die Vola-Normierung."),
    _qt_row(qt_w_lookback, "Lookback-Fenster (Kerzen) fuer die Wilks-Toleranzgrenzen."),
    _qt_row(qt_w_p_content, "Geforderter abgedeckter Anteil des Toleranzintervalls."),
    _qt_row(qt_w_exit_lookback, "Eigenes Fenster (Kerzen) fuer den Exit-Mittelwert des Preis-Spreads."),
    _qt_row(qt_w_confidence, "Konfidenzniveau des Toleranzintervalls."),
    _qt_row(qt_w_recalib_every, "Rekalibrierungs-Intervall (Kerzen)."),
    _qt_row(qt_w_max_holding_hours, "Maximale Haltedauer eines Trades (Stunden)."),
    _qt_row(qt_w_max_concurrent, "Max. gleichzeitig offene Positionen (fuer die Equity-Kurve)."),
    _qt_row(qt_w_trade_mode, "'max_only' (Paper-Original) oder 'all_significant'."),
    _qt_row(qt_w_fee_bps, "Gebuehren in Basispunkten je Leg."),
    qt_run_button,
    qt_output,
])
display(qt_ui)


In [5]:
# =====================================================================
# Abschnitt 4: Schnelltest -- Callback
# =====================================================================
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def _qt_on_run_clicked(_button=None):
    with qt_output:
        clear_output(wait=True)

        if qt_w_start.value is None:
            print("Bitte 'Start' setzen.")
            return

        start_ts = pd.Timestamp(qt_w_start.value, tz="UTC")
        end_ts = start_ts + pd.Timedelta(days=qt_w_days.value)

        cliques_df_qt = cliques_by_variant[qt_w_variant.value]
        cliques_df_qt = cliques_df_qt.loc[cliques_df_qt["basket"] == qt_w_basket.value]

        max_holding_candles_qt = max(1, int(round(qt_w_max_holding_hours.value * 60 / 5)))

        print(f"Schnelltest laeuft -- Basket: {qt_w_basket.value} | Zeitraum: {start_ts.date()} -> {end_ts.date()} "
              f"({qt_w_days.value} Tage) | ewma_decay={qt_w_ewma_decay.value} lookback={qt_w_lookback.value} "
              f"p_content={qt_w_p_content.value} trade_mode={qt_w_trade_mode.value} ...\n")

        trades_df_qt = pte.run_pairtrading_backtest(
            cliques_df=cliques_df_qt, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
            timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
            start_date=start_ts, end_date=end_ts - pd.Timedelta(days=1),
            decay=qt_w_ewma_decay.value, lookback_candles=qt_w_lookback.value,
            p_content=qt_w_p_content.value, confidence=qt_w_confidence.value,
            exit_lookback_candles=qt_w_exit_lookback.value,
            max_holding_candles=max_holding_candles_qt, fee_bps=qt_w_fee_bps.value,
            trade_mode=qt_w_trade_mode.value, recalib_every=qt_w_recalib_every.value,
        )
        qt_last_result.clear()
        qt_last_result.update(trades_df=trades_df_qt, cliques_df=cliques_df_qt)

        if trades_df_qt.empty:
            print("Keine Trades in diesem kurzen Zeitraum mit diesen Parametern -- "
                  "Zeitraum/Basket wechseln oder Parameter lockern (z.B. p_content senken).")
            return

        equity_curve_qt = pte.compute_equity_curve(
            trades_df_qt, initial_capital=tb.FIXED_INITIAL_CAPITAL,
            max_concurrent_positions=qt_w_max_concurrent.value,
        )
        stats_qt = pte.compute_backtest_stats(trades_df_qt, equity_curve_qt)

        stats_display_qt = pd.DataFrame([{
            "Anzahl Trades": stats_qt["n_trades"],
            "Win-Rate": f"{stats_qt['win_rate']:.1%}",
            "Sharpe": f"{stats_qt['sharpe']:.2f}" if np.isfinite(stats_qt["sharpe"]) else "n/a",
            "Max Drawdown": f"{stats_qt['max_drawdown']:.2%}" if np.isfinite(stats_qt["max_drawdown"]) else "n/a",
            "Gesamtrendite": f"{stats_qt['total_return']:.2%}" if np.isfinite(stats_qt["total_return"]) else "n/a",
            "Durchschnittliche Haltedauer": f"{stats_qt['avg_holding_hours']:.1f}h" if np.isfinite(stats_qt["avg_holding_hours"]) else "n/a",
        }]).T
        stats_display_qt.columns = ["Wert"]
        display(stats_display_qt)

        # Signalplot je Clique-Discovery-Tag im Zeitraum (i.d.R. 1 Tag bei
        # wenigen Tagen Dauer) -- dieselbe Signalkette wie im Backtest, mit
        # Warmup-Vorlauf fuer EWMA/Toleranzgrenzen (siehe Abschnitt 5c).
        discovery_days_qt = sorted(cliques_df_qt.loc[
            (cliques_df_qt["discovery_date"] >= start_ts) & (cliques_df_qt["discovery_date"] < end_ts),
            "discovery_date",
        ].unique())
        if not discovery_days_qt:
            print("Keine Clique-Discovery in diesem Zeitraum -- kein Signalplot moeglich.")
        price_matrix_qt = ohlcv_df.pivot_table(
            index=TIMESTAMP_COL, columns=ASSET_ID_COL, values=CLOSE_COL, aggfunc="last",
        ).sort_index()
        ts_index_qt = price_matrix_qt.index
        warmup_candles_qt = qt_w_lookback.value + 30

        for day in discovery_days_qt:
            day = pd.Timestamp(day)
            clique_row = cliques_df_qt.loc[cliques_df_qt["discovery_date"] == day].iloc[0]
            clique_ids = [a for a in clique_row["asset_ids"] if a in price_matrix_qt.columns]
            if len(clique_ids) < 2:
                continue

            day_start_pos = ts_index_qt.searchsorted(day.normalize())
            day_end_pos = ts_index_qt.searchsorted(day.normalize() + pd.Timedelta(days=1))
            window_start_pos = max(0, day_start_pos - warmup_candles_qt)
            if day_start_pos - warmup_candles_qt < 0 or day_end_pos <= day_start_pos:
                print(f"{day.date()}: zu wenig Historie fuer Warmup ({warmup_candles_qt} Kerzen) -- uebersprungen.")
                continue

            window_prices_qt = price_matrix_qt.iloc[window_start_pos:day_end_pos][clique_ids]
            spread_df_qt = build_pairwise_return_spreads(window_prices_qt)
            adjusted_df_qt = compute_adjusted_spreads(spread_df_qt, decay=qt_w_ewma_decay.value)
            max_spread_df_qt = compute_max_adjusted_spread(adjusted_df_qt)
            tolerance_df_qt = generate_tolerance_signals(
                max_spread_df_qt, lookback_candles=qt_w_lookback.value,
                p_content=qt_w_p_content.value, confidence=qt_w_confidence.value,
                recalib_every=qt_w_recalib_every.value,
            )

            day_trades_qt = trades_df_qt.loc[trades_df_qt["discovery_date"] == day.normalize()]
            print(f"\n{day.date()} -- Basket {qt_w_basket.value} -- {len(clique_ids)} Clique-Mitglieder, "
                  f"{len(day_trades_qt)} Trade(s)")
            pte.plot_xmax_with_tolerance_bands(max_spread_df_qt, tolerance_df_qt, day_trades_qt)

            # Preis-Spread je gehandeltem Paar mit Entry-/Exit-Markern, analog
            # Abschnitt 5c (Trade-Explorer) -- hedge_ratio fix auf 1.0, siehe
            # dortige Begruendung (In-Sample-OLS wuerde die Tagesbewegung
            # wegregressieren). Nur Paare mit >=1 Trade an diesem Tag, sonst
            # waere das Grid bei grossen Cliquen unuebersichtlich.
            traded_pairs_qt = sorted(day_trades_qt["pair"].astype(str).unique())
            if traded_pairs_qt:
                day_start_qt = day.normalize()
                day_end_qt = day_start_qt + pd.Timedelta(days=1)
                day_prices_qt = price_matrix_qt.loc[
                    (price_matrix_qt.index >= day_start_qt) & (price_matrix_qt.index < day_end_qt), clique_ids
                ].dropna(how="any")
                symbols_qt = [id_to_symbol.get(a, f"ID:{a}") for a in clique_ids]
                day_prices_qt.columns = symbols_qt

                if len(day_prices_qt) < 10:
                    print(f"  Nur {len(day_prices_qt)} vollstaendige Kerzen an diesem Tag -- Spread-Plot uebersprungen.")
                else:
                    n_pairs_qt = len(traded_pairs_qt)
                    n_cols_qt = min(3, n_pairs_qt)
                    n_rows_qt = int(np.ceil(n_pairs_qt / n_cols_qt))
                    fig_qt, axes_qt = plt.subplots(n_rows_qt, n_cols_qt, figsize=(4.5 * n_cols_qt, 3.2 * n_rows_qt), squeeze=False)

                    for pi, pair_str_qt in enumerate(traded_pairs_qt):
                        ax_qt = axes_qt[pi // n_cols_qt][pi % n_cols_qt]
                        pair_trades_qt = day_trades_qt.loc[day_trades_qt["pair"].astype(str) == pair_str_qt]
                        t1_qt, t2_qt = pair_trades_qt.iloc[0]["asset_long_sym"], pair_trades_qt.iloc[0]["asset_short_sym"]
                        if t1_qt not in day_prices_qt.columns or t2_qt not in day_prices_qt.columns:
                            ax_qt.axis("off")
                            continue
                        spread_qt = compute_price_spread(day_prices_qt, t1_qt, t2_qt, 1.0)
                        ax_qt.plot(spread_qt.index, spread_qt.values, linewidth=1, color="steelblue")
                        ax_qt.axhline(spread_qt.mean(), color="grey", linestyle="--", linewidth=0.8)

                        exit_reasons_qt = []
                        for _, trade_qt in pair_trades_qt.iterrows():
                            if trade_qt["entry_time"] in spread_qt.index:
                                ax_qt.axvline(trade_qt["entry_time"], color="firebrick", linewidth=1, alpha=0.7)
                                ax_qt.scatter([trade_qt["entry_time"]], [spread_qt.loc[trade_qt["entry_time"]]], color="firebrick", zorder=5, s=25)
                            if trade_qt["exit_time"] in spread_qt.index:
                                ax_qt.axvline(trade_qt["exit_time"], color="seagreen", linewidth=1, alpha=0.7)
                                ax_qt.scatter([trade_qt["exit_time"]], [spread_qt.loc[trade_qt["exit_time"]]], color="seagreen", zorder=5, s=25, marker="s")
                            exit_reasons_qt.append(trade_qt["exit_reason"])

                        title_qt = f"{t1_qt}-{t2_qt}"
                        if exit_reasons_qt:
                            title_qt += f" ({len(exit_reasons_qt)} Trade(s), Exit: {', '.join(exit_reasons_qt)})"
                        ax_qt.set_title(title_qt, fontsize=9)
                        ax_qt.tick_params(axis="x", rotation=45, labelsize=7)
                        ax_qt.tick_params(axis="y", labelsize=7)

                    for j in range(n_pairs_qt, n_rows_qt * n_cols_qt):
                        axes_qt[j // n_cols_qt][j % n_cols_qt].axis("off")

                    legend_handles_qt = [
                        Line2D([0], [0], color="firebrick", marker="o", linestyle="-", markersize=5, label="Entry"),
                        Line2D([0], [0], color="seagreen", marker="s", linestyle="-", markersize=5, label="Exit"),
                    ]
                    fig_qt.legend(handles=legend_handles_qt, loc="upper right", fontsize=8)
                    fig_qt.suptitle(f"Preis-Spreads -- {qt_w_basket.value} -- {day.date()}")
                    plt.tight_layout()
                    plt.show()

        display(trades_df_qt[[
            "pair", "asset_long_sym", "asset_short_sym", "direction", "entry_time", "exit_time",
            "exit_reason", "gross_return", "net_return", "holding_candles",
        ]].reset_index(drop=True))


qt_run_button.on_click(_qt_on_run_clicked)


# 5. Optuna-Tuning (walk-forward, persistenter Storage + Dashboard)

Der EINZIGE Tuning-Weg in diesem Notebook (bewusst so reduziert -- kein Nebeneinander
mehrerer Varianten): nutzt [Optuna](https://optuna.org/) (TPE-Sampler) statt Grid-Search.
Fuer jeden Trial laeuft der volle Backtest (`pairtrading_engine.run_pairtrading_backtest`)
ueber ALLE Walk-Forward-Validierungsfenster, nicht nur ueber einen einzelnen Zeitraum.
Zielgroesse: mittlere Validierungs-Rendite ueber die Fenster (Robustheit statt
Glueckstreffer, siehe `tune_backtest._objective_metric`).


**Zeitliche Trennung (kritisch, siehe `pairtrading_engine.generate_walkforward_windows`):**
der Gesamtzeitraum wird in drei Bloecke zerlegt:

1. **Trainings-/Validierungszeitraum**, in dem Optuna sucht (walk-forward: mehrere
   aufeinanderfolgende Train/Val-Fenster).
2. Ein **Embargo-Puffer** zwischen Validierung und finalem Test, um Leakage aus
   ueberlappenden Rekalibrierungsfenstern zu vermeiden.
3. Ein **finaler Testzeitraum**, der waehrend des GESAMTEN Tunings unberuehrt bleibt und erst
   am Ende, EINMAL, mit den final gewaehlten Parametern durchlaufen wird (letzte Zelle dieses
   Abschnitts). `pairtrading_engine.assert_no_test_period_leakage` prueft dies vor JEDEM
   einzelnen Trial und bricht mit einer klaren Fehlermeldung ab, falls der Suchraum den
   Testzeitraum beruehrt.

**Widget-Bedienung dieser drei Bloecke:** um eine widerspruechliche manuelle Datumsauswahl
(z.B. Test-Zeitraum VOR Train/Val, oder ueberlappende Zeitraeume) strukturell auszuschliessen,
werden Train/Val-Zeitraum und Test-Zeitraum NICHT mehr einzeln eingegeben. Stattdessen waehlt
man nur EINEN Gesamtzeitraum (Gesamt-Start/Gesamt-Ende), einen Testanteil in Prozent (welcher
Anteil des Gesamtzeitraums, gerechnet vom ENDE her, als Test reserviert wird) und den
Embargo-Puffer in Tagen -- die vier tatsaechlich verwendeten Datumswerte (Train/Val
Start/Ende, Test Start/Ende) werden daraus automatisch berechnet und nur noch (read-only) zur
Kontrolle angezeigt.

**Theoriegeleitete Filter vs. tunbare Signalparameter** (methodische Nachvollziehbarkeit,
siehe Ernie Chan, Kapitel Backtesting, zu Data-Snooping-Bias -- ausfuehrliche Begruendung im
Docstring von `tune_backtest.py`):
- **NICHT** Teil der Optuna-Suche: `min_persistence` (H0-Persistenzschwelle der
  Vietoris-Rips-Clique-Discovery, `02_tda_clustering/helperfunctions.py`) sowie ADF-autolag,
  KPSS-regression, FDR-alpha (Stationaritaets-/Mean-Reversion-Diagnostik auf
  Preis-Level-Spreads, `03_signal_generation/TradeData.ipynb`) -- diese bestimmen, WELCHE
  Cliquen/Paare ueberhaupt als handelbar gelten, BEVOR irgendein Backtest beginnt. Es sind
  literaturbasierte statistische Vortests mit klaren Konventionen, keine Hyperparameter einer
  Handelsstrategie -- sie ex-post gegen die Sharpe Ratio zu optimieren waere klassisches
  Data-Snooping.
- **Teil** der Optuna-Suche: `lookback_candles`, `p_content`, `trade_mode`, `recalib_every`,
  `max_holding_hours`, `max_concurrent_positions` -- reine Signal-/Trade-Management-Parameter
  INNERHALB von `run_pairtrading_backtest`, nachdem die Cliquen/Paare bereits feststehen.

**Persistenter SQLite-Storage + optuna-dashboard:** standardmaessig aktiviert (Checkbox
"Persistenter SQLite-Storage" unten). Die Study wird in einer SQLite-Datei
(`tune_backtest.DEFAULT_STORAGE_PATH`, per Widget aenderbar) gespeichert, ueberlebt
Notebook-/Kernel-Neustarts (`load_if_exists=True` -- ein erneuter Lauf mit demselben
Study-Namen haengt weitere Trials an, statt bei 0 neu zu beginnen), und kann WAEHREND das
Tuning laeuft parallel von `optuna-dashboard` gelesen werden:

```
optuna-dashboard sqlite:///<Pfad zur .db-Datei, siehe SQLite-Datei-Feld unten> --port 8080 --server wsgiref
```

**Windows-Hinweis zum Dashboard-Server:** `--server wsgiref` ist noetig, weil der Optuna-Dashboard-Default (`gunicorn`) auf Windows mit `ModuleNotFoundError: No module named 'fcntl'` fehlschlaegt (gunicorn ist Unix-only) -- getestet und bestaetigt, `wsgiref` (Python-Standardbibliothek, plattformunabhaengig) funktioniert fehlerfrei als Ersatz.

Dann im Browser: **http://127.0.0.1:8080** -- zeigt Parameter-Importance, Parallel-Coordinate-
Plot und Optimization-History live, aktualisiert sich automatisch waehrend neue Trials
abgeschlossen werden. Mehrere Studies (z.B. je Basket) landen in derselben `.db`-Datei und
werden im Dashboard einzeln zur Auswahl gelistet.

**Windows-Hinweis:** `tune_backtest.py` setzt beim Import `KMP_DUPLICATE_LIB_OK=TRUE`
(bekanntes OpenMP/DLL-Problem bei mehreren numpy/scipy-Runtimes im selben Prozessbaum,
relevant falls `study.optimize(..., n_jobs>1)` fuer parallele Trials genutzt wird -- hier
nicht standardmaessig aktiv, aber vom Aufrufer waehlbar).

Ruft ausschliesslich bereits vorhandene Bausteine aus `pairtrading_engine.py` auf
(`run_pairtrading_backtest` wird NICHT veraendert, nur wiederverwendet) -- die eigentliche
Tuning-Logik liegt in `tune_backtest.py`.


In [6]:
# =====================================================================
# Abschnitt 5: Widgets fuer das Optuna-Tuning
# =====================================================================
# tune_backtest.py enthaelt die gesamte Tuning-Logik (Suchraum, Zielgroesse,
# Pruning, persistenter Storage) -- dieses Notebook baut nur die Bedienober-
# flaeche darueber und ruft tb.tune_pairtrading_strategy() im Callback
# (naechste Zelle) auf. Siehe Modul-Docstring von tune_backtest.py fuer die
# vollstaendige Begruendung aller Design-Entscheidungen.

# --- Universum: welche Cliquen-Variante und welcher Basket wird getunt? ---
# "raw" vs. "residual" siehe Abschnitt 2 (Cliquen-Cache laden).
ot_w_variant = widgets.Dropdown(options=["raw", "residual"], value="raw", description="Cliquen-Variante:", style={"description_width": "160px"})
ot_w_basket = widgets.Dropdown(options=basket_labels_available, value=basket_labels_available[0], description="Basket:", style={"description_width": "160px"})

# --- Zeitliche Trennung: EIN Gesamtzeitraum + Testanteil (%) + Embargo ---
# Statt vier freien Datumsfeldern (Train/Val Start/Ende, Test Start/Ende),
# die widerspruechliche Kombinationen zulassen wuerden (z.B. Test VOR
# Train/Val, oder ueberlappende Zeitraeume), wird hier NUR EIN Gesamt-
# zeitraum gewaehlt und Train/Val/Test WERDEN DARAUS BERECHNET:
#   [Gesamt-Start, Gesamt-Ende)
#     = [Train/Val-Block][Embargo-Puffer][Test-Block, am Ende, test_pct %]
# Das macht eine falsche Datumsauswahl strukturell unmoeglich -- es gibt
# keine Freiheitsgrade mehr, die man widerspruechlich einstellen koennte.
# Die drei Bloecke aus dem Modul-Docstring von tune_backtest.py bleiben
# inhaltlich identisch:
# 1) [Train/Val Start, Train/Val Ende): der Bereich, INNERHALB dessen Optuna
#    Walk-Forward-Fenster bildet und sucht (siehe Walk-Forward-Widgets unten).
# 2) Embargo-Puffer (Tage) zwischen Train/Val Ende und Test Start: verhindert,
#    dass Rekalibrierungs-Lookback-Fenster des finalen Tests in den Trainings-
#    zeitraum hineinreichen (Leakage-Schutz).
# 3) [Test Start, Test Ende): bleibt WAEHREND DES GESAMTEN TUNINGS unberuehrt
#    und wird erst in Abschnitt 5b (ganz am Ende, EINMAL) verwendet.
# pairtrading_engine.assert_no_test_period_leakage prueft zusaetzlich vor
# JEDEM Trial, dass kein Walk-Forward-Fenster in [Test Start, Test Ende)
# hineinreicht -- doppelte Absicherung on top der strukturellen Loesung hier.
ot_w_overall_start = widgets.DatePicker(description="Gesamt-Start:", style={"description_width": "160px"})
ot_w_overall_end = widgets.DatePicker(description="Gesamt-Ende:", style={"description_width": "160px"})
ot_w_test_pct = widgets.IntSlider(value=15, min=5, max=50, description="Testanteil (%):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
ot_w_embargo_days = widgets.IntSlider(value=5, min=0, max=30, description="Embargo (Tage):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))

# Read-only Anzeige der DARAUS berechneten Zeitraeume -- nicht editierbar,
# nur zur Kontrolle, was tatsaechlich verwendet wird.
ot_w_train_val_start = widgets.DatePicker(description="-> Train/Val Start:", disabled=True, style={"description_width": "160px"})
ot_w_train_val_end = widgets.DatePicker(description="-> Train/Val Ende:", disabled=True, style={"description_width": "160px"})
ot_w_test_start = widgets.DatePicker(description="-> Test Start:", disabled=True, style={"description_width": "160px"})
ot_w_test_end = widgets.DatePicker(description="-> Test Ende:", disabled=True, style={"description_width": "160px"})


def _ot_default_overall_range():
    """Default-Gesamtzeitraum: Januar-Februar 2025 (2 Monate) -- klein
    gehalten, damit ein Testlauf schnell durchlaeuft (siehe Chat-Diskussion
    zu Ladezeiten). Faellt auf [data_min, data_max] zurueck, falls dieser
    Bereich ausserhalb der geladenen Daten liegt."""
    fixed_start = pd.Timestamp("2025-01-01")
    fixed_end = pd.Timestamp("2025-03-01")
    data_min_ts = pd.Timestamp(data_min)
    data_max_ts = pd.Timestamp(data_max)
    if fixed_start >= data_min_ts and fixed_end <= data_max_ts:
        return fixed_start.date(), fixed_end.date()
    return data_min, data_max


_ot_os, _ot_oe = _ot_default_overall_range()
ot_w_overall_start.value = _ot_os
ot_w_overall_end.value = _ot_oe


def _ot_recompute_split(change=None):
    """Berechnet Train/Val- und Test-Zeitraum aus Gesamtzeitraum, Testanteil
    (%) und Embargo (Tage) neu -- bei JEDER Aenderung eines der drei Eingabe-
    Widgets (siehe .observe()-Aufrufe unten). Rechnet mit pd.Timestamp statt
    datetime.date, weil `date + pd.Timedelta` einen date (kein Timestamp)
    zurueckgibt und somit KEIN .date() kennt -- erst am Ende wird auf
    .date() konvertiert."""
    if ot_w_overall_start.value is None or ot_w_overall_end.value is None:
        return
    overall_start = pd.Timestamp(ot_w_overall_start.value)
    overall_end = pd.Timestamp(ot_w_overall_end.value)
    if overall_end <= overall_start:
        return

    total_days = (overall_end - overall_start).days
    test_days = max(1, int(round(total_days * ot_w_test_pct.value / 100)))
    embargo_days = ot_w_embargo_days.value
    test_end = overall_end

    # Train/Val bekommt den Rest, NACHDEM Test-Anteil und Embargo abgezogen
    # sind -- garantiert train_val_end + embargo <= test_start, keine
    # Ueberlappung moeglich.
    train_val_start = overall_start
    train_val_end = overall_end - pd.Timedelta(days=test_days + embargo_days)
    if train_val_end <= train_val_start:
        # Testanteil + Embargo zu gross fuer den Gesamtzeitraum -- Train/Val
        # auf ein Minimum von 1 Tag klemmen statt negativ/leer werden zu
        # lassen (verhindert eine stille Fehlkonfiguration).
        train_val_end = train_val_start + pd.Timedelta(days=1)
    test_start = train_val_end + pd.Timedelta(days=embargo_days)
    if test_start > test_end:
        # Selbst nach dem Klemmen von train_val_end reicht der verbleibende
        # Platz nicht fuer den vollen Embargo-Puffer VOR test_end -- test_start
        # zusaetzlich auf test_end klemmen, damit NIE test_start > test_end
        # entstehen kann (sonst waere der Testzeitraum negativ/undefiniert).
        test_start = test_end

    ot_w_train_val_start.value = train_val_start.date()
    ot_w_train_val_end.value = train_val_end.date()
    ot_w_test_start.value = test_start.date()
    ot_w_test_end.value = test_end.date()


ot_w_overall_start.observe(_ot_recompute_split, names="value")
ot_w_overall_end.observe(_ot_recompute_split, names="value")
ot_w_test_pct.observe(_ot_recompute_split, names="value")
ot_w_embargo_days.observe(_ot_recompute_split, names="value")
_ot_recompute_split()

# --- Walk-Forward-Fenster INNERHALB von [Train/Val Start, Train/Val Ende) ---
# pairtrading_engine.generate_walkforward_windows baut daraus eine Liste
# aufeinanderfolgender (train, val)-Bloecke: train_days Training, dann
# wf_embargo_days Puffer, dann val_days Validierung, dann step_days weiter.
# Jeder Optuna-Trial wird (in Stufe 2, siehe ot-callback) ueber ALLE diese
# Validierungsfenster ausgewertet, nicht nur ueber eines -- das ist die
# Walk-Forward-Robustheitspruefung, siehe tune_backtest._objective_metric.
ot_w_train_days = widgets.IntSlider(value=30, min=5, max=90, description="Trainingsfenster (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ot_w_val_days = widgets.IntSlider(value=10, min=3, max=30, description="Validierungsfenster (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ot_w_step_days = widgets.IntSlider(value=15, min=3, max=60, description="Schrittweite (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ot_w_wf_embargo_days = widgets.IntSlider(value=2, min=0, max=20, description="WF-internes Embargo (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))

# --- Optuna-Steuerung ---
# n_trials = Anzahl NEUER Trials in DIESEM Lauf (nicht die Gesamtzahl der
# Study -- bei aktivem persistentem Storage kommen sie zu evtl. bereits
# vorhandenen Trials der Study HINZU, siehe Persistenz-Widgets unten).
# seed steuert den TPE-Sampler (nur bei einer NEU erstellten Study wirksam).
ot_w_n_trials = widgets.IntSlider(value=50, min=5, max=300, step=5, description="n_trials (NEUE Trials):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ot_w_seed = widgets.IntText(value=42, description="Seed:", style={"description_width": "160px"})

# --- Persistenter SQLite-Storage + optuna-dashboard ---
# Checkbox AN (Default): tb.create_or_load_study() baut/laedt eine Study
# gegen eine SQLite-Datei -- ueberlebt Notebook-/Kernel-Neustarts, mehrere
# Laeufe mit demselben Study-Namen haengen Trials AN statt neu zu beginnen,
# UND waehrend das Tuning laeuft kann `optuna-dashboard sqlite:///<Pfad>
# --server wsgiref` (Windows -- siehe Abschnitt 5 oben) parallel gestartet
# werden, um den Fortschritt live im Browser zu verfolgen.
# Checkbox AUS: reine In-Memory-Study, kein Dashboard, kein Ueberleben eines
# Neustarts -- nur fuer schnelle Tests ohne Dashboard-Bedarf gedacht.
# Study-Name leer = Basket-Name wird verwendet (siehe ot-callback).
ot_w_persistent = widgets.Checkbox(value=True, description="Persistenter SQLite-Storage (fuer optuna-dashboard)", indent=False)
ot_w_study_name = widgets.Text(value="", placeholder="Default: Basket-Name", description="Study-Name:", style={"description_width": "160px"})
ot_w_storage_path = widgets.Text(value=str(tb.DEFAULT_STORAGE_PATH), description="SQLite-Datei:", style={"description_width": "160px"}, layout=widgets.Layout(width="560px"))


# --- Suchraum-Parameter optional fixieren (statt von Optuna zu suchen) ---
# Fuer jeden der 6 tunbaren Parameter (siehe tune_backtest.tune_pairtrading_strategy-
# Docstring, Abschnitt "Suchraum") kann hier eine Checkbox aktiviert werden, um ihn
# auf EINEN festen Wert zu setzen -- dieser Parameter wird dann NICHT mehr von
# Optuna durchsucht (kein trial.suggest_* dafuer), sondern in jedem Trial exakt so
# verwendet. Reduziert den Suchraum gezielt, z.B. um trade_mode auf "max_only"
# festzunageln (exakte Paper-Replikation) waehrend der Rest weiter getunt wird.
ot_w_fix_lookback = widgets.Checkbox(value=False, description="fixieren", indent=False, layout=widgets.Layout(width="90px"))
ot_w_fix_lookback_value = widgets.Dropdown(options=[36, 72, 144, 288], value=144, description="lookback_candles:", style={"description_width": "160px"})
ot_w_fix_p_content = widgets.Checkbox(value=False, description="fixieren", indent=False, layout=widgets.Layout(width="90px"))
ot_w_fix_p_content_value = widgets.Dropdown(options=[0.90, 0.95], value=0.90, description="p_content:", style={"description_width": "160px"})
ot_w_fix_trade_mode = widgets.Checkbox(value=False, description="fixieren", indent=False, layout=widgets.Layout(width="90px"))
ot_w_fix_trade_mode_value = widgets.Dropdown(options=["max_only", "all_significant"], value="max_only", description="trade_mode:", style={"description_width": "160px"})
ot_w_fix_recalib_every = widgets.Checkbox(value=False, description="fixieren", indent=False, layout=widgets.Layout(width="90px"))
ot_w_fix_recalib_every_value = widgets.IntSlider(value=12, min=6, max=48, description="recalib_every:", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
ot_w_fix_max_holding_hours = widgets.Checkbox(value=False, description="fixieren", indent=False, layout=widgets.Layout(width="90px"))
ot_w_fix_max_holding_hours_value = widgets.FloatSlider(value=12.0, min=4.0, max=48.0, step=0.5, description="max_holding_hours:", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
ot_w_fix_max_concurrent = widgets.Checkbox(value=False, description="fixieren", indent=False, layout=widgets.Layout(width="90px"))
ot_w_fix_max_concurrent_value = widgets.IntSlider(value=10, min=3, max=15, description="max_concurrent_positions:", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
ot_w_fix_ewma_decay = widgets.Checkbox(value=False, description="fixieren", indent=False, layout=widgets.Layout(width="90px"))
ot_w_fix_ewma_decay_value = widgets.Dropdown(options=[0.89, 0.94, 0.97, 0.985], value=0.94, description="ewma_decay:", style={"description_width": "160px"})
ot_w_fix_exit_lookback = widgets.Checkbox(value=False, description="fixieren", indent=False, layout=widgets.Layout(width="90px"))
ot_w_fix_exit_lookback_value = widgets.Dropdown(options=[100, 200, 400, 600, 1000], value=400, description="exit_lookback_candles:", style={"description_width": "160px"})


def _ot_collect_fixed_params() -> dict:
    """Liest alle aktivierten 'fixieren'-Checkboxen aus und baut daraus das
    fixed_params-dict fuer tb.tune_pairtrading_strategy (siehe dortigen
    Docstring -- fixierte Parameter werden NICHT von Optuna gesucht)."""
    fixed = {}
    if ot_w_fix_lookback.value:
        fixed["lookback_candles"] = ot_w_fix_lookback_value.value
    if ot_w_fix_p_content.value:
        fixed["p_content"] = ot_w_fix_p_content_value.value
    if ot_w_fix_trade_mode.value:
        fixed["trade_mode"] = ot_w_fix_trade_mode_value.value
    if ot_w_fix_recalib_every.value:
        fixed["recalib_every"] = ot_w_fix_recalib_every_value.value
    if ot_w_fix_max_holding_hours.value:
        fixed["max_holding_hours"] = ot_w_fix_max_holding_hours_value.value
    if ot_w_fix_max_concurrent.value:
        fixed["max_concurrent_positions"] = ot_w_fix_max_concurrent_value.value
    if ot_w_fix_ewma_decay.value:
        fixed["ewma_decay"] = ot_w_fix_ewma_decay_value.value
    if ot_w_fix_exit_lookback.value:
        fixed["exit_lookback_candles"] = ot_w_fix_exit_lookback_value.value
    return fixed


ot_run_button = widgets.Button(description="Optuna-Tuning starten", button_style="primary", icon="play")
ot_output = widgets.Output()
ot_last_result = {}  # haelt TuningResult (siehe tune_backtest.py) fuer Abschnitt 5b

def _ot_row(widget, description):
    """Baut eine Zeile: Widget links, kurze graue Erklaerung rechts daneben."""
    desc_html = widgets.HTML(
        f"<span style='color:#888; font-size:0.9em;'>{description}</span>",
        layout=widgets.Layout(margin="6px 0 0 12px"),
    )
    return widgets.HBox([widget, desc_html])


ot_ui = widgets.VBox([
    widgets.HTML("<b>Universum</b>"),
    _ot_row(ot_w_variant, "Cliquen-Cache-Variante (siehe Abschnitt 2): 'raw' oder 'residual'."),
    _ot_row(ot_w_basket, "Welcher TDA-Basket (Cluster-Gruppe) getunt wird."),

    widgets.HTML("<b>Zeitliche Trennung</b> (EIN Gesamtzeitraum -- Train/Val/Test werden automatisch berechnet, keine widerspruechliche manuelle Auswahl mehr moeglich)"),
    _ot_row(widgets.HBox([ot_w_overall_start, ot_w_overall_end]),
            "Gesamter Zeitraum, der geladen/verwendet wird."),
    _ot_row(ot_w_test_pct,
            "Wie viel Prozent des Gesamtzeitraums (am ENDE) als finaler, unberuehrter Testzeitraum reserviert werden."),
    _ot_row(ot_w_embargo_days,
            "Puffer (Tage) zwischen Train/Val Ende und Test Start -- verhindert Datenleck durch ueberlappende Lookback-Fenster."),
    widgets.HTML("<span style='color:#aaa; font-size:0.85em;'>Automatisch berechnet (nicht editierbar):</span>"),
    _ot_row(widgets.HBox([ot_w_train_val_start, ot_w_train_val_end]),
            "Zeitraum, INNERHALB dessen Optuna sucht (Walk-Forward-Fenster darin, siehe unten)."),
    _ot_row(widgets.HBox([ot_w_test_start, ot_w_test_end]),
            "Bleibt WAEHREND DES GESAMTEN TUNINGS unberuehrt, wird erst in Abschnitt 5b EINMAL final getestet."),

    widgets.HTML("<b>Walk-Forward-Fenster (innerhalb Train/Val)</b>"),
    _ot_row(ot_w_train_days, "Laenge des Trainingsfensters je Walk-Forward-Schritt (Tage)."),
    _ot_row(ot_w_val_days, "Laenge des Validierungsfensters je Walk-Forward-Schritt (Tage) -- hierauf wird jeder Trial bewertet."),
    _ot_row(ot_w_step_days, "Um wie viele Tage das Fenster bei jedem Walk-Forward-Schritt weiterrueckt."),
    _ot_row(ot_w_wf_embargo_days, "Puffer (Tage) zwischen Training und Validierung INNERHALB eines Walk-Forward-Schritts."),

    widgets.HTML("<b>Optuna</b>"),
    _ot_row(ot_w_n_trials, "Anzahl NEUER Trials in diesem Lauf (kommen bei aktiver Persistenz zu bestehenden Trials hinzu)."),
    _ot_row(ot_w_seed, "Zufalls-Seed fuer den TPE-Sampler -- nur bei einer NEU erstellten Study wirksam."),

    widgets.HTML("<b>Suchraum-Parameter fixieren</b> (optional -- nicht angehakte Parameter werden wie gewohnt von Optuna gesucht)"),
    _ot_row(widgets.HBox([ot_w_fix_lookback, ot_w_fix_lookback_value]), "Wenn aktiv: lookback_candles wird NICHT gesucht, sondern immer auf diesen Wert gesetzt."),
    _ot_row(widgets.HBox([ot_w_fix_p_content, ot_w_fix_p_content_value]), "Wenn aktiv: p_content fixiert."),
    _ot_row(widgets.HBox([ot_w_fix_trade_mode, ot_w_fix_trade_mode_value]), "Wenn aktiv: trade_mode fixiert (z.B. 'max_only' fuer exakte Paper-Replikation)."),
    _ot_row(widgets.HBox([ot_w_fix_recalib_every, ot_w_fix_recalib_every_value]), "Wenn aktiv: recalib_every fixiert."),
    _ot_row(widgets.HBox([ot_w_fix_max_holding_hours, ot_w_fix_max_holding_hours_value]), "Wenn aktiv: max_holding_hours fixiert."),
    _ot_row(widgets.HBox([ot_w_fix_max_concurrent, ot_w_fix_max_concurrent_value]), "Wenn aktiv: max_concurrent_positions fixiert."),
    _ot_row(widgets.HBox([ot_w_fix_ewma_decay, ot_w_fix_ewma_decay_value]), "Wenn aktiv: ewma_decay fixiert (RiskMetrics-Default 0.94 ist fuer Tagesdaten kalibriert, bei 5-Min-Kerzen nicht per se richtig)."),
    _ot_row(widgets.HBox([ot_w_fix_exit_lookback, ot_w_fix_exit_lookback_value]), "Wenn aktiv: exit_lookback_candles fixiert (eigenes Fenster fuer den Exit-Mittelwert des Preis-Spreads)."),

    widgets.HTML("<b>Persistenz & Dashboard</b> (siehe Anleitung in Abschnitt 5 oben)"),
    _ot_row(ot_w_persistent, "An: Trials werden in einer SQLite-Datei gespeichert (ueberlebt Neustarts, mit optuna-dashboard live verfolgbar)."),
    _ot_row(ot_w_study_name, "Name der Optuna-Study. Leer = Basket-Name wird automatisch verwendet."),
    _ot_row(ot_w_storage_path, "Pfad zur SQLite-Datei, in der die Study gespeichert wird."),

    ot_run_button,
    ot_output,
])
display(ot_ui)


In [7]:
# =====================================================================
# Abschnitt 5: Callback -- startet einen Optuna-Tuning-Lauf ueber
# tune_backtest.tune_pairtrading_strategy() und zeigt die Ergebnisse an.
# =====================================================================
import matplotlib.pyplot as plt

def _ot_on_run_clicked(_button=None):
    with ot_output:
        clear_output(wait=True)

        # Pflichtfelder pruefen, BEVOR irgendein teurer Aufruf startet.
        for w, label in [
            (ot_w_train_val_start, "Train/Val Start"), (ot_w_train_val_end, "Train/Val Ende"),
            (ot_w_test_start, "Test Start"), (ot_w_test_end, "Test Ende"),
        ]:
            if w.value is None:
                print(f"Bitte '{label}' setzen.")
                return

        cliques_df = cliques_by_variant[ot_w_variant.value]
        cliques_df = cliques_df.loc[cliques_df["basket"] == ot_w_basket.value]

        # Study-Name-Default: Basket-Name, wenn das Textfeld leer gelassen wurde
        # (siehe ot-widgets) -- so hat jeder Basket standardmaessig seine eigene
        # Study, auch ohne dass explizit ein Name eingegeben werden muss.
        effective_study_name = ot_w_study_name.value or ot_w_basket.value
        print(f"Optuna-Tuning laeuft -- Basket: {ot_w_basket.value} | "
              f"Train/Val: {ot_w_train_val_start.value} -> {ot_w_train_val_end.value} | "
              f"Test (unberuehrt bis zum Schluss): {ot_w_test_start.value} -> {ot_w_test_end.value} | "
              f"n_trials={ot_w_n_trials.value} ...")
        if ot_w_persistent.value:
            print(f"Persistenter Storage: sqlite:///{ot_w_storage_path.value} | Study-Name: '{effective_study_name}' "
                  f"-- waehrend des Laufs mit optuna-dashboard live verfolgbar (siehe Anleitung oben).\n")
        else:
            print("Persistenter Storage: AUS (In-Memory, kein Dashboard).\n")


        # Der eigentliche Tuning-Lauf -- ALLE Logik (Suchraum, Backtest,
        # Zielgroesse, Walk-Forward, Persistenz) steckt in tune_backtest.py,
        # hier wird nur mit den Widget-Werten aufgerufen. ValueError/RuntimeError
        # werden gezielt abgefangen (siehe
        # tune_backtest.tune_pairtrading_strategy-Docstring): z.B. wenn die
        # Walk-Forward-Konfiguration kein einziges Fenster ergibt.
        try:
            result = tb.tune_pairtrading_strategy(
                basket=ot_w_basket.value, cliques_df=cliques_df, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
                timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
                train_val_start=ot_w_train_val_start.value, train_val_end=ot_w_train_val_end.value,
                test_start=ot_w_test_start.value, test_end=ot_w_test_end.value,
                train_days=ot_w_train_days.value, val_days=ot_w_val_days.value,
                step_days=ot_w_step_days.value, embargo_days=ot_w_wf_embargo_days.value,
                n_trials=ot_w_n_trials.value, seed=ot_w_seed.value,
                output_path=PROCESSED_DIR / "tuning" / f"optuna_trials_{ot_w_basket.value.replace(' ', '_')}.parquet",
                study_name=ot_w_study_name.value or None,
                storage_path=ot_w_storage_path.value,
                use_persistent_storage=ot_w_persistent.value,
                fixed_params=_ot_collect_fixed_params() or None,
            )
        except (ValueError, RuntimeError) as e:
            print(f"Tuning abgebrochen: {e}")
            return

        # fuer Abschnitt 5b (finale, einmalige Testzeitraum-Auswertung) merken.
        ot_last_result.clear()
        ot_last_result.update(result=result)

        tb.summarize_tuning_results([result])

        # --- Top-10-Tabelle aller abgeschlossenen Trials ---
        completed = result.trials_df.loc[result.trials_df["status"] == "completed"].sort_values("objective", ascending=False)
        if not completed.empty:
            print("\nTop 10 abgeschlossene Trials nach Objective:")
            display(completed.head(10)[[
                "lookback_candles", "p_content", "trade_mode", "recalib_every",
                "max_holding_hours", "max_concurrent_positions", "ewma_decay", "objective",
                "mean_fold_return", "std_fold_return",
            ]])


ot_run_button.on_click(_ot_on_run_clicked)

## 5b. Finale Testzeitraum-Auswertung (EINMALIG, nach Abschluss des Tunings)

Laeuft `run_pairtrading_backtest` mit den `best_params` aus Abschnitt 5 GENAU EINMAL auf dem
finalen Testzeitraum -- der bis zu diesem Punkt in KEINEM Trial angefasst wurde. Dies ist der
einzige Zeitpunkt, an dem der Testzeitraum ueberhaupt verwendet wird. Ergebnis hier ist die
Zahl, die in der Thesis als Out-of-Sample-Performance zitiert werden sollte -- nicht die
Validierungs-Objective-Werte aus Abschnitt 5 (die sind Teil der Modellwahl, nicht des Tests).


In [8]:
# =====================================================================
# Abschnitt 5b: EINMALIGE finale Auswertung auf dem unberuehrten Testzeitraum
# =====================================================================
# WICHTIG: diese Zelle sollte im gesamten Notebook nur EIN EINZIGES MAL nach
# Abschluss des Tunings ausgefuehrt werden. Sie verwendet best_params aus dem
# LETZTEN Optuna-Lauf (Abschnitt 5) und testet sie auf [Test Start, Test Ende)
# -- dem Zeitraum, der waehrend ALLER Trials in Abschnitt 5 nie angefasst
# wurde (siehe assert_no_test_period_leakage in pairtrading_engine.py).
# Das Ergebnis hier, NICHT die Validierungs-Objective-Werte aus Abschnitt 5,
# ist die Out-of-Sample-Performance-Zahl fuer die Thesis.
if not ot_last_result:
    print("Erst Abschnitt 5 (Optuna-Tuning) ausfuehren.")
else:
    result = ot_last_result["result"]
    best_params = result.best_params

    # Fixierte Parameter (siehe Abschnitt 5, "Suchraum-Parameter fixieren")
    # stehen NICHT in best_params -- Optuna kennt nur Parameter, die es
    # tatsaechlich gesucht hat (trial.suggest_*). Fuer jeden der 6 Parameter
    # daher zuerst best_params pruefen, sonst auf den im Fixier-Widget
    # eingestellten Wert zurueckfallen (_ot_collect_fixed_params, Abschnitt 5).
    effective_params = {**_ot_collect_fixed_params(), **best_params}
    missing = {"lookback_candles", "p_content", "trade_mode", "recalib_every",
               "max_holding_hours", "max_concurrent_positions", "ewma_decay",
               "exit_lookback_candles"} - set(effective_params)
    if missing:
        raise KeyError(
            f"Parameter {sorted(missing)} sind weder in best_params noch als fixiert "
            f"eingestellt -- Abschnitt 5 (Optuna-Tuning) erneut mit denselben "
            f"Fixier-Einstellungen ausfuehren oder die Haekchen in Abschnitt 5 pruefen."
        )

    print(f"Finale Testzeitraum-Auswertung -- Basket: {result.basket} | "
          f"Testzeitraum: {ot_w_test_start.value} -> {ot_w_test_end.value} | "
          f"effective_params={effective_params}\n")

    cliques_df_test = cliques_by_variant[ot_w_variant.value]
    cliques_df_test = cliques_df_test.loc[cliques_df_test["basket"] == result.basket]

    # max_holding_hours -> Kerzen umrechnen, exakt dieselbe Konvention wie im
    # Tuning-Callback (5-Min-Kerzen: 60 Minuten / 5 = 12 Kerzen/Stunde).
    max_holding_candles_test = max(1, int(round(effective_params["max_holding_hours"] * 60 / 5)))

    # run_pairtrading_backtest mit den getunten Signal-Parametern (best_params),
    # inkl. ewma_decay (seit Umbau Teil des Suchraums, siehe tune_backtest.py),
    # UND den weiterhin fest vorgegebenen, nicht getunten Parametern
    # (FIXED_CONFIDENCE, FIXED_FEE_BPS, FIXED_INITIAL_CAPITAL -- siehe Modul-
    # Docstring von tune_backtest.py, Abschnitt "Theoriegeleitete Filterparameter
    # vs. tunbare Signalparameter", fuer die Begruendung, warum diese Werte
    # fest sind statt Teil der Optuna-Suche zu sein).
    test_trades_df = pte.run_pairtrading_backtest(
        cliques_df=cliques_df_test, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
        timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
        start_date=ot_w_test_start.value, end_date=ot_w_test_end.value,
        decay=effective_params["ewma_decay"], lookback_candles=effective_params["lookback_candles"],
        p_content=effective_params["p_content"], confidence=tb.FIXED_CONFIDENCE,
        exit_lookback_candles=effective_params["exit_lookback_candles"],
        max_holding_candles=max_holding_candles_test, fee_bps=tb.FIXED_FEE_BPS,
        trade_mode=effective_params["trade_mode"], recalib_every=effective_params["recalib_every"],
    )
    test_equity_curve = pte.compute_equity_curve(
        test_trades_df, initial_capital=tb.FIXED_INITIAL_CAPITAL,
        max_concurrent_positions=effective_params["max_concurrent_positions"],
    )
    test_stats = pte.compute_backtest_stats(test_trades_df, test_equity_curve)

    # Fuer Abschnitt 5c (Trade-Explorer): test_trades_df UND den fuer diesen
    # Testlauf verwendeten Cliquen-Filter mit in ot_last_result ablegen, damit
    # der Explorer nicht selbst nochmal run_pairtrading_backtest aufrufen muss.
    ot_last_result["test_trades_df"] = test_trades_df
    ot_last_result["cliques_df_test"] = cliques_df_test
    ot_last_result["effective_params"] = effective_params

    if test_trades_df.empty:
        print("Keine Trades im finalen Testzeitraum mit den gewaehlten Parametern.")
    else:
        test_stats_display = pd.DataFrame([{
            "Anzahl Trades": test_stats["n_trades"],
            "Win-Rate": f"{test_stats['win_rate']:.1%}",
            "Sharpe": f"{test_stats['sharpe']:.2f}" if np.isfinite(test_stats["sharpe"]) else "n/a",
            "Max Drawdown": f"{test_stats['max_drawdown']:.2%}" if np.isfinite(test_stats["max_drawdown"]) else "n/a",
            "Gesamtrendite (finaler Test, out-of-sample)": f"{test_stats['total_return']:.2%}" if np.isfinite(test_stats["total_return"]) else "n/a",
            "Durchschnittliche Haltedauer": f"{test_stats['avg_holding_hours']:.1f}h" if np.isfinite(test_stats["avg_holding_hours"]) else "n/a",
        }]).T
        test_stats_display.columns = ["Wert"]
        display(test_stats_display)
        pte.plot_equity_curve(test_equity_curve, initial_capital=tb.FIXED_INITIAL_CAPITAL)


Erst Abschnitt 5 (Optuna-Tuning) ausfuehren.


## 5c. Trade-Explorer: Ein-/Ausstiegspunkte pro Tag (finaler Testzeitraum)

Zeigt die **tatsaechlichen Trades** aus Abschnitt 5b (`test_trades_df`, mit `best_params`
auf dem unberuehrten Testzeitraum simuliert) tageweise als Preis-Level-Spread-Plots mit
Entry-/Exit-Markern -- **keine eigene, vereinfachte Signalberechnung** wie im separaten
`spreadanalyse.ipynb`, sondern die echten Ergebnisse von `simulate_pair_trade`
(inkl. `max_holding_candles`, `trade_mode`, `exit_reason`).

Die Tradinglogik selbst wird hier nicht angefasst -- diese Zelle liest nur `test_trades_df`
aus Abschnitt 5b und `ohlcv_df` aus Abschnitt 1 und stellt sie dar. Muss NACH Abschnitt 5b
ausgefuehrt werden.

Marker:
- **Rot** = Entry (`entry_time`), **Grün** = Exit (`exit_time`).
- Exit-Grund im Subplot-Titel: `signal` (Rueckkehr zur Toleranzgrenze), `holding_limit`
  (Zeitlimit erreicht) oder `day_end` (Ende des verfuegbaren Preisfensters).

In [9]:
# =====================================================================
# Widgets: Tag-Navigation ueber die discovery_date-Werte in test_trades_df
# =====================================================================
te_w_day_label = widgets.HTML()
te_w_prev_button = widgets.Button(description="< Tag", layout=widgets.Layout(width="90px"))
te_w_next_button = widgets.Button(description="Tag >", layout=widgets.Layout(width="90px"))
te_output = widgets.Output()
te_state = {"day_idx": 0}

#beim sprad wird hier der normalisierte Spread (EWMA) fuer die Clique berechnet und geplottet, um zu zeigen, WARUM an den Entry-/Exit-Punkten tatsaechlich ein Signal entstand.
def _te_current_days():
    """Sortierte, eindeutige discovery_date-Werte MIT mindestens einem Trade
    in test_trades_df (siehe Abschnitt 5b) -- Tage ohne Trade werden uebersprungen,
    da es dort nichts zu markieren gaebe."""
    if not ot_last_result or "test_trades_df" not in ot_last_result:
        return pd.DatetimeIndex([])
    trades = ot_last_result["test_trades_df"]
    if trades.empty:
        return pd.DatetimeIndex([])
    return pd.DatetimeIndex(sorted(trades["discovery_date"].unique()))


def _te_render(_=None):
    with te_output:
        clear_output(wait=True)

        if not ot_last_result or "test_trades_df" not in ot_last_result:
            print("Erst Abschnitt 5b (finale Testzeitraum-Auswertung) ausfuehren.")
            return

        trades = ot_last_result["test_trades_df"]
        if trades.empty:
            print("test_trades_df ist leer -- keine Trades im finalen Testzeitraum.")
            return

        days = _te_current_days()
        te_state["day_idx"] = max(0, min(te_state["day_idx"], len(days) - 1))
        day = days[te_state["day_idx"]]
        te_w_day_label.value = f"<b>{day.date()}</b> &nbsp; (Tag {te_state['day_idx'] + 1} von {len(days)} mit Trade(s))"

        day_trades = trades.loc[trades["discovery_date"] == day]

        basket = ot_last_result["result"].basket
        cliques_df_day = cliques_by_variant[ot_w_variant.value]
        clique_row = cliques_df_day.loc[
            (cliques_df_day["basket"] == basket) & (cliques_df_day["discovery_date"] == day)
        ]
        if clique_row.empty:
            print(f"Keine Clique-Definition fuer {day.date()} gefunden (Basket {basket}).")
            return
        asset_ids = [a for a in clique_row.iloc[0]["asset_ids"] if a in ohlcv_df[ASSET_ID_COL].unique()]
        symbols = [id_to_symbol.get(a, f"ID:{a}") for a in asset_ids]
        if len(symbols) < 2:
            print(f"Clique an {day.date()} hat weniger als 2 handelbare Assets.")
            return

        print(f"Clique-Mitglieder ({len(symbols)}): {', '.join(symbols)} | "
              f"{len(day_trades)} Trade(s) an diesem Tag")

        day_start = day.normalize()
        day_end = day_start + pd.Timedelta(days=1)
        price_matrix_day = ohlcv_df.pivot_table(
            index=TIMESTAMP_COL, columns=ASSET_ID_COL, values=CLOSE_COL, aggfunc="last",
        ).sort_index()
        day_prices = price_matrix_day.loc[
            (price_matrix_day.index >= day_start) & (price_matrix_day.index < day_end), asset_ids
        ].dropna(how="any")
        day_prices.columns = symbols

        if len(day_prices) < 10:
            print(f"Nur {len(day_prices)} vollstaendige Kerzen an diesem Tag -- zu wenig fuer eine sinnvolle Darstellung.")
            return

        import matplotlib.pyplot as plt
        from itertools import combinations
        from matplotlib.lines import Line2D

        # EWMA-normierter Spread (Xt) je Paar + Toleranzgrenzen, MIT denselben
        # effective_params wie der Backtest (Abschnitt 5b) -- zeigt, WARUM an
        # den Entry-Punkten tatsaechlich ein Signal entstand. Braucht
        # Warmup-Vorlauf vor dem Tag (lookback_candles + 30 Kerzen), analog
        # run_pairtrading_backtest/spreadanalyse.ipynb._sa_compute_signal.
        eff = ot_last_result.get("effective_params")
        adjusted_df_day = None
        tolerance_df_day = None
        exit_mean_df_day = None
        price_spread_df_day = None
        if eff is not None:
            lookback_candles = eff["lookback_candles"]
            exit_lookback_candles = eff["exit_lookback_candles"]
            warmup_candles = max(lookback_candles, exit_lookback_candles) + 30
            ts_index = price_matrix_day.index
            day_start_pos = ts_index.searchsorted(day_start)
            day_end_pos = ts_index.searchsorted(day_end)
            window_start_pos = max(0, day_start_pos - warmup_candles)
            if day_start_pos - warmup_candles >= 0:
                window_prices = price_matrix_day.iloc[window_start_pos:day_end_pos][asset_ids]
                window_prices.columns = symbols
                window_prices = window_prices.dropna(how="any")
                if len(window_prices) >= warmup_candles + 10:
                    spread_df_day = build_pairwise_return_spreads(window_prices)
                    adjusted_df_day = compute_adjusted_spreads(spread_df_day, decay=eff["ewma_decay"])
                    max_spread_df_day = compute_max_adjusted_spread(adjusted_df_day)
                    tolerance_df_day = generate_tolerance_signals(
                        max_spread_df_day, lookback_candles=lookback_candles,
                        p_content=eff["p_content"], confidence=tb.FIXED_CONFIDENCE,
                        recalib_every=eff["recalib_every"],
                    )
                    day_slice_day = adjusted_df_day.index[adjusted_df_day.index >= day_start]
                    tolerance_df_day = tolerance_df_day.loc[day_slice_day]
                    adjusted_df_day = adjusted_df_day.loc[day_slice_day]
                    # Spalten-Tupel sind in der Reihenfolge von window_prices.columns
                    # (nicht zwingend alphabetisch, siehe build_pairwise_return_spreads-
                    # Docstring) -- auf sortierte Tupel normalisieren, damit der Such-
                    # Key tuple(sorted((t1, t2))) unten immer trifft.
                    adjusted_df_day.columns = [tuple(sorted(c)) for c in adjusted_df_day.columns]

                    # Exit-Signal: PREIS-Level-Spread + eigener rollierender
                    # Mittelwert (exit_lookback_candles), siehe
                    # pairtrading_engine.simulate_pair_trade -- unabhaengig
                    # vom EWMA-Entry-Signal oben. hedge_lookback_candles
                    # default = exit_lookback_candles (siehe run_pairtrading_backtest).
                    hedge_lookback_candles = eff.get("hedge_lookback_candles", exit_lookback_candles)
                    price_spread_df_day, _hedge_ratios_day = build_pairwise_price_spreads(
                        window_prices, hedge_lookback_candles=hedge_lookback_candles,
                    )
                    exit_mean_df_day = compute_exit_mean(
                        price_spread_df_day, exit_lookback_candles=exit_lookback_candles,
                    )
                    price_spread_df_day = price_spread_df_day.loc[day_slice_day]
                    exit_mean_df_day = exit_mean_df_day.loc[day_slice_day]
                    price_spread_df_day.columns = [tuple(sorted(c)) for c in price_spread_df_day.columns]
                    exit_mean_df_day.columns = [tuple(sorted(c)) for c in exit_mean_df_day.columns]

        pairs = list(combinations(symbols, 2))
        n_pairs = len(pairs)
        n_cols = min(2, n_pairs)
        n_rows = int(np.ceil(n_pairs / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(8.5 * n_cols, 5.5 * n_rows), squeeze=False)

        for i, (t1, t2) in enumerate(pairs):
            ax = axes[i // n_cols][i % n_cols]
            pair_key = tuple(sorted((t1, t2)))

            # Preis-Spread MIT rollierendem Exit-Mittelwert, falls verfuegbar
            # (echte Exit-Signalgroesse -- siehe simulate_pair_trade). Fallback
            # auf den statischen Tages-Spread (kein Warmup), falls effective_params
            # fehlen (z.B. Abschnitt 5b noch nicht gelaufen).
            if price_spread_df_day is not None and pair_key in price_spread_df_day.columns:
                spread = price_spread_df_day[pair_key]
                exit_mean = exit_mean_df_day[pair_key]
                ax.plot(spread.index, spread.values, linewidth=1, color="steelblue", label="Preis-Spread")
                ax.plot(exit_mean.index, exit_mean.values, color="#1baf7a", linewidth=1.2, linestyle="-.", label="Exit-Mittelwert")
            else:
                spread = compute_price_spread(day_prices, t1, t2, 1.0)
                ax.plot(spread.index, spread.values, linewidth=1, color="steelblue", label="Preis-Spread")
                ax.axhline(spread.mean(), color="grey", linestyle="--", linewidth=0.8, label="Mittelwert (statisch, kein Warmup)")

            # EWMA-normierter Spread Xt DIESES Paares + Toleranzgrenzen auf einer
            # zweiten y-Achse -- zeigt, wie nah Xt an der ENTRY-Grenze war, ohne
            # die Preis-Spread-Skala zu verzerren (voellig unterschiedliche
            # Groessenordnungen: Log-Preis-Differenz vs. EWMA-normierter Wert um 0).
            if adjusted_df_day is not None and pair_key in adjusted_df_day.columns:
                ax2 = ax.twinx()
                xt = adjusted_df_day[pair_key]
                ax2.plot(xt.index, xt.values, linewidth=0.9, color="darkorange", alpha=0.8, label="Xt (EWMA)")
                ax2.plot(tolerance_df_day.index, tolerance_df_day["upper_limit"], linewidth=0.8, color="firebrick", linestyle=":", alpha=0.7)
                ax2.plot(tolerance_df_day.index, -tolerance_df_day["upper_limit"], linewidth=0.8, color="firebrick", linestyle=":", alpha=0.7)
                ax2.axhline(0, color="darkorange", linewidth=0.5, alpha=0.3)
                ax2.tick_params(axis="y", labelsize=6, labelcolor="darkorange")
                ax2.set_ylabel("Xt", fontsize=7, color="darkorange")

            pair_syms = {t1, t2}
            pair_trades = day_trades.loc[
                day_trades.apply(lambda r: {r["asset_long_sym"], r["asset_short_sym"]} == pair_syms, axis=1)
            ]
            exit_reasons = []
            for _, trade in pair_trades.iterrows():
                if trade["entry_time"] in spread.index:
                    ax.scatter([trade["entry_time"]], [spread.loc[trade["entry_time"]]], color="firebrick", zorder=5, s=35, alpha=0.75)
                if trade["exit_time"] in spread.index:
                    ax.scatter([trade["exit_time"]], [spread.loc[trade["exit_time"]]], color="seagreen", zorder=5, s=35, marker="s", alpha=0.75)
                exit_reasons.append(trade["exit_reason"])

            title = f"{t1}-{t2}"
            if exit_reasons:
                reason_counts = pd.Series(exit_reasons).value_counts()
                reason_str = ", ".join(f"{r}×{c}" for r, c in reason_counts.items())
                title += f" ({len(exit_reasons)} Trade(s), Exit: {reason_str})"
            ax.set_title(title, fontsize=9)
            ax.tick_params(axis="x", rotation=45, labelsize=7)
            ax.tick_params(axis="y", labelsize=7)

        for j in range(n_pairs, n_rows * n_cols):
            axes[j // n_cols][j % n_cols].axis("off")

        legend_handles = [
            Line2D([0], [0], color="firebrick", marker="o", linestyle="-", markersize=5, label="Entry"),
            Line2D([0], [0], color="seagreen", marker="s", linestyle="-", markersize=5, label="Exit"),
            Line2D([0], [0], color="#1baf7a", linestyle="-.", linewidth=1.2, label="Exit-Mittelwert"),
            Line2D([0], [0], color="darkorange", linestyle="-", linewidth=1.2, label="Xt (EWMA, rechte Achse)"),
            Line2D([0], [0], color="firebrick", linestyle=":", linewidth=1.2, label="Entry-Toleranzgrenze"),
        ]
        fig.legend(handles=legend_handles, loc="upper right", fontsize=8)
        fig.suptitle(f"Trades -- {basket} ({ot_w_variant.value}) -- {day.date()}")
        plt.tight_layout()
        plt.show()

        display(day_trades[[
            "pair", "asset_long_sym", "asset_short_sym", "direction", "entry_time", "exit_time",
            "exit_reason", "gross_return", "net_return", "holding_candles",
        ]].reset_index(drop=True))


def _te_on_prev(_button=None):
    te_state["day_idx"] -= 1
    _te_render()


def _te_on_next(_button=None):
    te_state["day_idx"] += 1
    _te_render()


te_w_prev_button.on_click(_te_on_prev)
te_w_next_button.on_click(_te_on_next)

te_ui = widgets.VBox([
    widgets.HBox([te_w_prev_button, te_w_day_label, te_w_next_button]),
    te_output,
])
display(te_ui)
_te_render()

# 5d. Schnelles, breites Tuning (kein Walk-Forward, groessere Suchbereiche)

Ergaenzung zu Abschnitt 5 (praezise, aber langsam) fuer die **erste Exploration**:
statt eines Walk-Forward-Backtests ueber mehrere Fenster laeuft hier **EIN** Backtest
pro Trial direkt auf einem Trainingszeitraum -- deutlich schneller, dafuer weniger
robust gegen Overfitting auf genau diesen Zeitraum. Zielgroesse ist die **Sharpe
Ratio** dieses einen Backtests (statt der mittleren Walk-Forward-Rendite in
Abschnitt 5).

**Breiterer Suchraum** (siehe `fast_tune.tune_fast`-Docstring): dieselben sieben
Parameter wie in Abschnitt 5, dazu NEU `confidence` (dort fest auf 0.95), und alle
kontinuierlich/int statt nur ueber wenige kategoriale Stuetzstellen gesampelt --
deckt bei gleicher Trial-Anzahl einen groesseren Bereich ab.

**Zeitliche Trennung:** ein einfacher Train/Test-Split (`fast_tune.split_train_test`)
statt Walk-Forward-Fenstern -- EIN Trainingszeitraum zum Tunen, EIN separater
Testzeitraum am Ende zur groben Kontrolle (Abschnitt 5e).

**Wann diesen Tuner statt Abschnitt 5 nutzen:** um schnell viele Baskets/Zeitraeume
grob durchzuprobieren und ein Gefuehl zu bekommen, wo ueberhaupt Potential steckt --
bevor man das teurere, praezisere Walk-Forward-Tuning (Abschnitt 5) gezielt darauf
ansetzt. Das Ergebnis hier ist **keine** verlaessliche Out-of-Sample-Kennzahl fuer
die Thesis (dafuer weiterhin Abschnitt 5b).

In [10]:
# =====================================================================
# Abschnitt 5d: Schnelles Tuning -- Widgets
# =====================================================================
import fast_tune as ft

ft_w_variant = widgets.Dropdown(options=["raw", "residual"], value="raw", description="Cliquen-Variante:", style={"description_width": "160px"})
ft_w_basket = widgets.Dropdown(options=basket_labels_available, value=basket_labels_available[0], description="Basket:", style={"description_width": "160px"})

ft_w_overall_start = widgets.DatePicker(description="Gesamt-Start:", style={"description_width": "160px"})
ft_w_overall_end = widgets.DatePicker(description="Gesamt-Ende:", style={"description_width": "160px"})
ft_w_test_pct = widgets.IntSlider(value=20, min=5, max=50, description="Testanteil (%):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))

ft_w_train_start = widgets.DatePicker(description="-> Train Start:", disabled=True, style={"description_width": "160px"})
ft_w_train_end = widgets.DatePicker(description="-> Train Ende:", disabled=True, style={"description_width": "160px"})
ft_w_test_start = widgets.DatePicker(description="-> Test Start:", disabled=True, style={"description_width": "160px"})
ft_w_test_end = widgets.DatePicker(description="-> Test Ende:", disabled=True, style={"description_width": "160px"})


def _ft_default_overall_range():
    """Wie _ot_default_overall_range (Abschnitt 5) -- Default Januar-Maerz 2025,
    faellt auf [data_min, data_max] zurueck falls ausserhalb der geladenen Daten."""
    fixed_start = pd.Timestamp("2025-01-01")
    fixed_end = pd.Timestamp("2025-03-01")
    data_min_ts = pd.Timestamp(data_min)
    data_max_ts = pd.Timestamp(data_max)
    if fixed_start >= data_min_ts and fixed_end <= data_max_ts:
        return fixed_start.date(), fixed_end.date()
    return data_min, data_max


_ft_os, _ft_oe = _ft_default_overall_range()
ft_w_overall_start.value = _ft_os
ft_w_overall_end.value = _ft_oe


def _ft_recompute_split(change=None):
    if ft_w_overall_start.value is None or ft_w_overall_end.value is None:
        return
    overall_start = pd.Timestamp(ft_w_overall_start.value)
    overall_end = pd.Timestamp(ft_w_overall_end.value)
    if overall_end <= overall_start:
        return
    train, test = ft.split_train_test(overall_start, overall_end, test_fraction=ft_w_test_pct.value / 100)
    ft_w_train_start.value = train["start"].date()
    ft_w_train_end.value = train["end"].date()
    ft_w_test_start.value = test["start"].date()
    ft_w_test_end.value = test["end"].date()


ft_w_overall_start.observe(_ft_recompute_split, names="value")
ft_w_overall_end.observe(_ft_recompute_split, names="value")
ft_w_test_pct.observe(_ft_recompute_split, names="value")
_ft_recompute_split()

ft_w_n_trials = widgets.IntSlider(value=150, min=10, max=1000, step=10, description="n_trials (NEUE Trials):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ft_w_seed = widgets.IntText(value=42, description="Seed:", style={"description_width": "160px"})

ft_w_persistent = widgets.Checkbox(value=True, description="Persistenter SQLite-Storage (fuer optuna-dashboard)", indent=False)
ft_w_study_name = widgets.Text(value="", placeholder="Default: <Basket>_fast", description="Study-Name:", style={"description_width": "160px"})
ft_w_storage_path = widgets.Text(value=str(ft.DEFAULT_STORAGE_PATH), description="SQLite-Datei:", style={"description_width": "160px"}, layout=widgets.Layout(width="560px"))

ft_run_button = widgets.Button(description="Schnelles Tuning starten", button_style="warning", icon="bolt")
ft_output = widgets.Output()
ft_last_result = {}


def _ft_row(widget, description):
    desc_html = widgets.HTML(
        f"<span style=\'color:#888; font-size:0.9em;\'>{description}</span>",
        layout=widgets.Layout(margin="6px 0 0 12px"),
    )
    return widgets.HBox([widget, desc_html])


ft_ui = widgets.VBox([
    widgets.HTML("<b>Universum</b>"),
    _ft_row(ft_w_variant, "Cliquen-Cache-Variante (siehe Abschnitt 2): 'raw' oder 'residual'."),
    _ft_row(ft_w_basket, "Welcher TDA-Basket (Cluster-Gruppe) getunt wird."),

    widgets.HTML("<b>Zeitliche Trennung</b> (einfacher Train/Test-Split, KEIN Walk-Forward)"),
    _ft_row(widgets.HBox([ft_w_overall_start, ft_w_overall_end]), "Gesamter Zeitraum, der geladen/verwendet wird."),
    _ft_row(ft_w_test_pct, "Wie viel Prozent des Gesamtzeitraums (am ENDE) als Testzeitraum reserviert werden."),
    widgets.HTML("<span style=\'color:#aaa; font-size:0.85em;\'>Automatisch berechnet (nicht editierbar):</span>"),
    _ft_row(widgets.HBox([ft_w_train_start, ft_w_train_end]), "Trainingszeitraum, auf dem JEDER Trial direkt (ohne Walk-Forward) simuliert wird."),
    _ft_row(widgets.HBox([ft_w_test_start, ft_w_test_end]), "Bleibt waehrend des Tunings unberuehrt, wird erst in Abschnitt 5e ausgewertet."),

    widgets.HTML("<b>Optuna</b>"),
    _ft_row(ft_w_n_trials, "Anzahl NEUER Trials -- da JE EIN Backtest pro Trial (kein Walk-Forward), sind deutlich mehr Trials in aehnlicher Zeit moeglich als in Abschnitt 5."),
    _ft_row(ft_w_seed, "Zufalls-Seed fuer den TPE-Sampler -- nur bei einer NEU erstellten Study wirksam."),

    widgets.HTML("<b>Persistenz & Dashboard</b>"),
    _ft_row(ft_w_persistent, "An: eigene SQLite-Datei (getrennt von Abschnitt 5, siehe fast_tune.DEFAULT_STORAGE_PATH), live mit optuna-dashboard verfolgbar."),
    _ft_row(ft_w_study_name, "Name der Optuna-Study. Leer = '<Basket>_fast'."),
    _ft_row(ft_w_storage_path, "Pfad zur SQLite-Datei, in der die Study gespeichert wird."),

    ft_run_button,
    ft_output,
])
display(ft_ui)

In [11]:
# =====================================================================
# Abschnitt 5d: Callback -- startet einen schnellen Tuning-Lauf ueber
# fast_tune.tune_fast() und zeigt die Ergebnisse an.
# =====================================================================
import matplotlib.pyplot as plt


def _ft_on_run_clicked(_button=None):
    with ft_output:
        clear_output(wait=True)

        for w, label in [
            (ft_w_train_start, "Train Start"), (ft_w_train_end, "Train Ende"),
            (ft_w_test_start, "Test Start"), (ft_w_test_end, "Test Ende"),
        ]:
            if w.value is None:
                print(f"Bitte '{label}' setzen.")
                return

        cliques_df_ft = cliques_by_variant[ft_w_variant.value]
        cliques_df_ft = cliques_df_ft.loc[cliques_df_ft["basket"] == ft_w_basket.value]

        effective_study_name = ft_w_study_name.value or f"{ft_w_basket.value}_fast"
        print(f"Schnelles Tuning laeuft -- Basket: {ft_w_basket.value} | "
              f"Train: {ft_w_train_start.value} -> {ft_w_train_end.value} | "
              f"Test (unberuehrt bis Abschnitt 5e): {ft_w_test_start.value} -> {ft_w_test_end.value} | "
              f"n_trials={ft_w_n_trials.value} ...")
        if ft_w_persistent.value:
            print(f"Persistenter Storage: sqlite:///{ft_w_storage_path.value} | Study-Name: '{effective_study_name}'\n")
        else:
            print("Persistenter Storage: AUS (In-Memory, kein Dashboard).\n")

        result = ft.tune_fast(
            basket=ft_w_basket.value, cliques_df=cliques_df_ft, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
            timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
            train_start=ft_w_train_start.value, train_end=ft_w_train_end.value,
            n_trials=ft_w_n_trials.value, seed=ft_w_seed.value,
            output_path=PROCESSED_DIR / "tuning" / f"fast_trials_{ft_w_basket.value.replace(' ', '_')}.parquet",
            study_name=ft_w_study_name.value or None,
            storage_path=ft_w_storage_path.value,
            use_persistent_storage=ft_w_persistent.value,
        )
        ft_last_result.clear()
        ft_last_result.update(
            result=result, cliques_df=cliques_df_ft,
            test_start=ft_w_test_start.value, test_end=ft_w_test_end.value,
        )

        ft.summarize_fast_tuning_results([result])

        completed = result.trials_df.sort_values("sharpe", ascending=False)
        if not completed.empty:
            print("\nTop 10 Trials nach Sharpe:")
            display(completed.head(10)[[
                "lookback_candles", "p_content", "confidence", "trade_mode", "recalib_every",
                "max_holding_hours", "max_concurrent_positions", "ewma_decay",
                "sharpe", "total_return", "n_trades", "win_rate",
            ]])

            fig, ax = plt.subplots(figsize=(7, 4.5))
            finite = completed.loc[np.isfinite(completed["sharpe"])]
            ax.scatter(finite["n_trades"], finite["sharpe"], c=finite["total_return"], cmap="RdYlGn")
            ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
            ax.set_xlabel("Anzahl Trades")
            ax.set_ylabel("Sharpe (Objective)")
            ax.set_title(f"Trials: Sharpe vs. Trade-Anzahl, Farbe=total_return ({ft_w_basket.value})")
            plt.tight_layout()
            plt.show()


ft_run_button.on_click(_ft_on_run_clicked)

## 5e. Test-Auswertung des schnellen Tunings (grobe Kontrolle)

Wertet `best_params` aus Abschnitt 5d EINMAL auf dem waehrend des schnellen
Tunings unberuehrten Testzeitraum aus -- analog Abschnitt 5b, aber ohne
Walk-Forward-Kontext. Dient nur der **groben** Kontrolle ("bricht das
Ergebnis komplett zusammen, sobald wir den Trainingszeitraum verlassen?"),
NICHT als Ersatz fuer die praezisere Out-of-Sample-Auswertung in Abschnitt 5b.

In [12]:
# =====================================================================
# Abschnitt 5e: Test-Auswertung des schnellen Tunings
# =====================================================================
if not ft_last_result:
    print("Erst Abschnitt 5d (schnelles Tuning) ausfuehren.")
else:
    ft_best_params = ft_last_result["result"].best_params
    print(f"Test-Auswertung (schnelles Tuning) -- Basket: {ft_last_result['result'].basket} | "
          f"Testzeitraum: {ft_last_result['test_start']} -> {ft_last_result['test_end']} | "
          f"best_params={ft_best_params}\n")

    ft_test_result = ft.evaluate_on_test(
        best_params=ft_best_params, basket=ft_last_result["result"].basket,
        cliques_df=ft_last_result["cliques_df"], ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
        timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
        test_start=ft_last_result["test_start"], test_end=ft_last_result["test_end"],
    )
    ft_test_stats = ft_test_result["stats"]

    if ft_test_result["trades_df"].empty:
        print("Keine Trades im Testzeitraum mit den gewaehlten Parametern.")
    else:
        ft_test_stats_display = pd.DataFrame([{
            "Anzahl Trades": ft_test_stats["n_trades"],
            "Win-Rate": f"{ft_test_stats['win_rate']:.1%}",
            "Sharpe": f"{ft_test_stats['sharpe']:.2f}" if np.isfinite(ft_test_stats["sharpe"]) else "n/a",
            "Max Drawdown": f"{ft_test_stats['max_drawdown']:.2%}" if np.isfinite(ft_test_stats["max_drawdown"]) else "n/a",
            "Gesamtrendite (Test)": f"{ft_test_stats['total_return']:.2%}" if np.isfinite(ft_test_stats["total_return"]) else "n/a",
            "Durchschnittliche Haltedauer": f"{ft_test_stats['avg_holding_hours']:.1f}h" if np.isfinite(ft_test_stats["avg_holding_hours"]) else "n/a",
        }]).T
        ft_test_stats_display.columns = ["Wert"]
        display(ft_test_stats_display)
        pte.plot_equity_curve(ft_test_result["equity_curve"], initial_capital=ft.FIXED_INITIAL_CAPITAL)

Erst Abschnitt 5d (schnelles Tuning) ausfuehren.


# 6. Bollinger-Band-Variante (Preis-Level-Spread, gleitender Durchschnitt)

Dritte, PARALLELE Strategie neben Abschnitt 5 (EWMA-Toleranzintervall nach Lin, Chen &
Syu 2021, Renditedifferenz) und dem schnellen Tuner (Abschnitt 5d). Nutzt denselben
Cliquen-Cache und dieselben OHLCV-Daten, aber eine klassische Bollinger-Band-Signalkette
(`src/signals/bollinger.py`, `bollinger_pairtrading_engine.py`):

1. **Preis-Level-Spread** statt Renditedifferenz: `spread_t = log(P_i,t) - hedge_ratio *
   log(P_j,t)` (dieselbe Groesse wie in der ADF/KPSS-Stationaritaetsdiagnostik,
   `src.signals.spread.compute_price_spread`) -- die Hedge-Ratio wird per OLS ueber ein
   eigenes Lookback-Fenster geschaetzt.
2. **Gleitender Mittelwert + gleitende Standardabweichung** dieses Spreads ueber
   `lookback_candles` Kerzen -> Bollinger-Baender = Mittelwert +/- `n_std` * Std.
3. **Zmax_t** = groesster |Z-Score| ueber alle Paare der Clique -- macht Paare
   vergleichbar, analog zu Xmax_t in Abschnitt 5.
4. **Entry** bei Bandausbruch (`|Zt| > n_std`), **Exit** sobald der Preis-Spread
   seinen EIGENEN rollierenden Mittelwert (`exit_lookback_candles`) erreicht/kreuzt,
   oder Holding-Limit.

**Wie in Abschnitt 5**: eigener Optuna-Tuner (`tune_bollinger.py`,
Walk-Forward-Backtest), eigene persistente SQLite-Study
(`bollinger_study.db`, getrennt von `optuna_study.db`/`fast_tune_study.db`), damit die
drei Strategien direkt via `pairtrading_engine.plot_strategy_comparison` gegenuebergestellt
werden koennen, ohne sich gegenseitig im selben Storage zu vermischen.

In [13]:
# =====================================================================
# Abschnitt 6: Bollinger-Band-Tuning -- Widgets
# =====================================================================
import bollinger_pairtrading_engine as bpe
import tune_bollinger as tbo

bo_w_variant = widgets.Dropdown(options=["raw", "residual"], value="raw", description="Cliquen-Variante:", style={"description_width": "160px"})
bo_w_basket = widgets.Dropdown(options=basket_labels_available, value=basket_labels_available[0], description="Basket:", style={"description_width": "160px"})

bo_w_overall_start = widgets.DatePicker(description="Gesamt-Start:", style={"description_width": "160px"})
bo_w_overall_end = widgets.DatePicker(description="Gesamt-Ende:", style={"description_width": "160px"})
bo_w_test_pct = widgets.IntSlider(value=15, min=5, max=50, description="Testanteil (%):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
bo_w_embargo_days = widgets.IntSlider(value=5, min=0, max=30, description="Embargo (Tage):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))

bo_w_train_val_start = widgets.DatePicker(description="-> Train/Val Start:", disabled=True, style={"description_width": "160px"})
bo_w_train_val_end = widgets.DatePicker(description="-> Train/Val Ende:", disabled=True, style={"description_width": "160px"})
bo_w_test_start = widgets.DatePicker(description="-> Test Start:", disabled=True, style={"description_width": "160px"})
bo_w_test_end = widgets.DatePicker(description="-> Test Ende:", disabled=True, style={"description_width": "160px"})


def _bo_default_overall_range():
    """Wie _ot_default_overall_range (Abschnitt 5)."""
    fixed_start = pd.Timestamp("2025-01-01")
    fixed_end = pd.Timestamp("2025-03-01")
    data_min_ts = pd.Timestamp(data_min)
    data_max_ts = pd.Timestamp(data_max)
    if fixed_start >= data_min_ts and fixed_end <= data_max_ts:
        return fixed_start.date(), fixed_end.date()
    return data_min, data_max


_bo_os, _bo_oe = _bo_default_overall_range()
bo_w_overall_start.value = _bo_os
bo_w_overall_end.value = _bo_oe


def _bo_recompute_split(change=None):
    """Wie _ot_recompute_split (Abschnitt 5): Train/Val + Embargo + Test aus
    Gesamtzeitraum und Testanteil berechnen."""
    if bo_w_overall_start.value is None or bo_w_overall_end.value is None:
        return
    overall_start = pd.Timestamp(bo_w_overall_start.value)
    overall_end = pd.Timestamp(bo_w_overall_end.value)
    if overall_end <= overall_start:
        return

    total_days = (overall_end - overall_start).days
    test_days = max(1, int(round(total_days * bo_w_test_pct.value / 100)))
    embargo_days = bo_w_embargo_days.value
    test_end = overall_end

    train_val_start = overall_start
    train_val_end = overall_end - pd.Timedelta(days=test_days + embargo_days)
    if train_val_end <= train_val_start:
        train_val_end = train_val_start + pd.Timedelta(days=1)
    test_start = train_val_end + pd.Timedelta(days=embargo_days)
    if test_start > test_end:
        test_start = test_end

    bo_w_train_val_start.value = train_val_start.date()
    bo_w_train_val_end.value = train_val_end.date()
    bo_w_test_start.value = test_start.date()
    bo_w_test_end.value = test_end.date()


bo_w_overall_start.observe(_bo_recompute_split, names="value")
bo_w_overall_end.observe(_bo_recompute_split, names="value")
bo_w_test_pct.observe(_bo_recompute_split, names="value")
bo_w_embargo_days.observe(_bo_recompute_split, names="value")
_bo_recompute_split()

bo_w_train_days = widgets.IntSlider(value=30, min=5, max=90, description="Trainingsfenster (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
bo_w_val_days = widgets.IntSlider(value=10, min=3, max=30, description="Validierungsfenster (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
bo_w_step_days = widgets.IntSlider(value=15, min=3, max=60, description="Schrittweite (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
bo_w_wf_embargo_days = widgets.IntSlider(value=2, min=0, max=20, description="WF-internes Embargo (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))

bo_w_n_trials = widgets.IntSlider(value=50, min=5, max=300, step=5, description="n_trials (NEUE Trials):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
bo_w_seed = widgets.IntText(value=42, description="Seed:", style={"description_width": "160px"})

bo_w_persistent = widgets.Checkbox(value=True, description="Persistenter SQLite-Storage (fuer optuna-dashboard)", indent=False)
bo_w_study_name = widgets.Text(value="", placeholder="Default: <Basket>_bollinger", description="Study-Name:", style={"description_width": "160px"})
bo_w_storage_path = widgets.Text(value=str(tbo.DEFAULT_STORAGE_PATH), description="SQLite-Datei:", style={"description_width": "160px"}, layout=widgets.Layout(width="560px"))

bo_run_button = widgets.Button(description="Bollinger-Tuning starten", button_style="info", icon="play")
bo_output = widgets.Output()
bo_last_result = {}


def _bo_row(widget, description):
    desc_html = widgets.HTML(
        f"<span style=\'color:#888; font-size:0.9em;\'>{description}</span>",
        layout=widgets.Layout(margin="6px 0 0 12px"),
    )
    return widgets.HBox([widget, desc_html])


bo_ui = widgets.VBox([
    widgets.HTML("<b>Universum</b>"),
    _bo_row(bo_w_variant, "Cliquen-Cache-Variante (siehe Abschnitt 2): 'raw' oder 'residual'."),
    _bo_row(bo_w_basket, "Welcher TDA-Basket (Cluster-Gruppe) getunt wird."),

    widgets.HTML("<b>Zeitliche Trennung</b> (EIN Gesamtzeitraum -- Train/Val/Test werden automatisch berechnet)"),
    _bo_row(widgets.HBox([bo_w_overall_start, bo_w_overall_end]), "Gesamter Zeitraum, der geladen/verwendet wird."),
    _bo_row(bo_w_test_pct, "Wie viel Prozent des Gesamtzeitraums (am ENDE) als finaler, unberuehrter Testzeitraum reserviert werden."),
    _bo_row(bo_w_embargo_days, "Puffer (Tage) zwischen Train/Val Ende und Test Start."),
    widgets.HTML("<span style=\'color:#aaa; font-size:0.85em;\'>Automatisch berechnet (nicht editierbar):</span>"),
    _bo_row(widgets.HBox([bo_w_train_val_start, bo_w_train_val_end]), "Zeitraum, INNERHALB dessen Optuna sucht (Walk-Forward-Fenster darin)."),
    _bo_row(widgets.HBox([bo_w_test_start, bo_w_test_end]), "Bleibt waehrend des gesamten Tunings unberuehrt, wird erst unten (Abschnitt 6b) verwendet."),

    widgets.HTML("<b>Walk-Forward-Fenster (innerhalb Train/Val)</b>"),
    _bo_row(bo_w_train_days, "Laenge des Trainingsfensters je Walk-Forward-Schritt (Tage)."),
    _bo_row(bo_w_val_days, "Laenge des Validierungsfensters je Walk-Forward-Schritt (Tage)."),
    _bo_row(bo_w_step_days, "Um wie viele Tage das Fenster bei jedem Walk-Forward-Schritt weiterrueckt."),
    _bo_row(bo_w_wf_embargo_days, "Puffer (Tage) zwischen Training und Validierung INNERHALB eines Walk-Forward-Schritts."),

    widgets.HTML("<b>Optuna</b>"),
    _bo_row(bo_w_n_trials, "Anzahl NEUER Trials in diesem Lauf."),
    _bo_row(bo_w_seed, "Zufalls-Seed fuer den TPE-Sampler -- nur bei einer NEU erstellten Study wirksam."),

    widgets.HTML("<b>Persistenz & Dashboard</b>"),
    _bo_row(bo_w_persistent, "An: eigene SQLite-Datei (getrennt von Abschnitt 5/5d), live mit optuna-dashboard verfolgbar."),
    _bo_row(bo_w_study_name, "Name der Optuna-Study. Leer = '<Basket>_bollinger'."),
    _bo_row(bo_w_storage_path, "Pfad zur SQLite-Datei, in der die Study gespeichert wird."),

    bo_run_button,
    bo_output,
])
display(bo_ui)

In [14]:
# =====================================================================
# Abschnitt 6: Callback -- startet einen Bollinger-Tuning-Lauf ueber
# tune_bollinger.tune_bollinger_strategy() und zeigt die Ergebnisse an.
# =====================================================================
import matplotlib.pyplot as plt


def _bo_on_run_clicked(_button=None):
    with bo_output:
        clear_output(wait=True)

        for w, label in [
            (bo_w_train_val_start, "Train/Val Start"), (bo_w_train_val_end, "Train/Val Ende"),
            (bo_w_test_start, "Test Start"), (bo_w_test_end, "Test Ende"),
        ]:
            if w.value is None:
                print(f"Bitte '{label}' setzen.")
                return

        cliques_df_bo = cliques_by_variant[bo_w_variant.value]
        cliques_df_bo = cliques_df_bo.loc[cliques_df_bo["basket"] == bo_w_basket.value]

        effective_study_name = bo_w_study_name.value or f"{bo_w_basket.value}_bollinger"
        print(f"Bollinger-Tuning laeuft -- Basket: {bo_w_basket.value} | "
              f"Train/Val: {bo_w_train_val_start.value} -> {bo_w_train_val_end.value} | "
              f"Test (unberuehrt bis Abschnitt 6b): {bo_w_test_start.value} -> {bo_w_test_end.value} | "
              f"n_trials={bo_w_n_trials.value} ...")
        if bo_w_persistent.value:
            print(f"Persistenter Storage: sqlite:///{bo_w_storage_path.value} | Study-Name: '{effective_study_name}'\n")
        else:
            print("Persistenter Storage: AUS (In-Memory, kein Dashboard).\n")


        try:
            result = tbo.tune_bollinger_strategy(
                basket=bo_w_basket.value, cliques_df=cliques_df_bo, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
                timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
                train_val_start=bo_w_train_val_start.value, train_val_end=bo_w_train_val_end.value,
                test_start=bo_w_test_start.value, test_end=bo_w_test_end.value,
                train_days=bo_w_train_days.value, val_days=bo_w_val_days.value,
                step_days=bo_w_step_days.value, embargo_days=bo_w_wf_embargo_days.value,
                n_trials=bo_w_n_trials.value, seed=bo_w_seed.value,
                output_path=PROCESSED_DIR / "tuning" / f"bollinger_trials_{bo_w_basket.value.replace(' ', '_')}.parquet",
                study_name=bo_w_study_name.value or None,
                storage_path=bo_w_storage_path.value,
                use_persistent_storage=bo_w_persistent.value,
            )
        except (ValueError, RuntimeError) as e:
            print(f"Tuning abgebrochen: {e}")
            return

        bo_last_result.clear()
        bo_last_result.update(
            result=result, cliques_df=cliques_df_bo,
            test_start=bo_w_test_start.value, test_end=bo_w_test_end.value,
        )

        tbo.summarize_bollinger_tuning_results([result])

        completed = result.trials_df.loc[result.trials_df["status"] == "completed"].sort_values("objective", ascending=False)
        if not completed.empty:
            print("\nTop 10 abgeschlossene Trials nach Objective:")
            display(completed.head(10)[[
                "lookback_candles", "n_std", "exit_lookback_candles", "trade_mode",
                "max_holding_hours", "max_concurrent_positions", "hedge_lookback_candles",
                "objective", "mean_fold_return", "std_fold_return",
            ]])


bo_run_button.on_click(_bo_on_run_clicked)

## 6b. Finale Testzeitraum-Auswertung (Bollinger-Band-Variante)

Wie Abschnitt 5b, aber fuer die Bollinger-Band-Strategie: laeuft `run_bollinger_backtest`
mit den `best_params` aus Abschnitt 6 GENAU EINMAL auf dem finalen Testzeitraum, der
waehrend des gesamten Tunings nie angefasst wurde.

In [15]:
# =====================================================================
# Abschnitt 6b: EINMALIGE finale Auswertung auf dem unberuehrten Testzeitraum
# =====================================================================
if not bo_last_result:
    print("Erst Abschnitt 6 (Bollinger-Tuning) ausfuehren.")
else:
    bo_result = bo_last_result["result"]
    bo_best_params = bo_result.best_params

    print(f"Finale Testzeitraum-Auswertung (Bollinger) -- Basket: {bo_result.basket} | "
          f"Testzeitraum: {bo_last_result['test_start']} -> {bo_last_result['test_end']} | "
          f"best_params={bo_best_params}\n")

    bo_max_holding_candles_test = max(1, int(round(bo_best_params["max_holding_hours"] * 60 / 5)))

    bo_test_trades_df = bpe.run_bollinger_backtest(
        cliques_df=bo_last_result["cliques_df"], ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
        timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
        start_date=bo_last_result["test_start"], end_date=bo_last_result["test_end"],
        lookback_candles=bo_best_params["lookback_candles"], n_std=bo_best_params["n_std"],
        exit_lookback_candles=bo_best_params["exit_lookback_candles"], max_holding_candles=bo_max_holding_candles_test,
        fee_bps=tbo.FIXED_FEE_BPS, trade_mode=bo_best_params["trade_mode"],
        hedge_lookback_candles=bo_best_params["hedge_lookback_candles"],
    )
    bo_test_equity_curve = pte.compute_equity_curve(
        bo_test_trades_df, initial_capital=tbo.FIXED_INITIAL_CAPITAL,
        max_concurrent_positions=bo_best_params["max_concurrent_positions"],
    )
    bo_test_stats = pte.compute_backtest_stats(bo_test_trades_df, bo_test_equity_curve)

    bo_last_result["test_trades_df"] = bo_test_trades_df
    bo_last_result["best_params"] = bo_best_params

    if bo_test_trades_df.empty:
        print("Keine Trades im finalen Testzeitraum mit den gewaehlten Parametern.")
    else:
        bo_test_stats_display = pd.DataFrame([{
            "Anzahl Trades": bo_test_stats["n_trades"],
            "Win-Rate": f"{bo_test_stats['win_rate']:.1%}",
            "Sharpe": f"{bo_test_stats['sharpe']:.2f}" if np.isfinite(bo_test_stats["sharpe"]) else "n/a",
            "Max Drawdown": f"{bo_test_stats['max_drawdown']:.2%}" if np.isfinite(bo_test_stats["max_drawdown"]) else "n/a",
            "Gesamtrendite (finaler Test, out-of-sample)": f"{bo_test_stats['total_return']:.2%}" if np.isfinite(bo_test_stats["total_return"]) else "n/a",
            "Durchschnittliche Haltedauer": f"{bo_test_stats['avg_holding_hours']:.1f}h" if np.isfinite(bo_test_stats["avg_holding_hours"]) else "n/a",
        }]).T
        bo_test_stats_display.columns = ["Wert"]
        display(bo_test_stats_display)
        pte.plot_equity_curve(bo_test_equity_curve, initial_capital=tbo.FIXED_INITIAL_CAPITAL)

        display(bo_test_trades_df[[
            "pair", "asset_long_sym", "asset_short_sym", "direction", "entry_time", "exit_time",
            "exit_reason", "gross_return", "net_return", "holding_candles",
        ]].reset_index(drop=True))

Erst Abschnitt 6 (Bollinger-Tuning) ausfuehren.


## 6c. Trade-Explorer: Preis-Spread mit Bollinger-Baendern (finaler Testzeitraum)

Wie Abschnitt 5c, aber fuer die Bollinger-Band-Strategie: zeigt die **tatsaechlichen
Trades** aus Abschnitt 6b (`test_trades_df`, mit `best_params` auf dem unberuehrten
Testzeitraum simuliert) tageweise als Preis-Spread-Plots MIT den Bollinger-Baendern
(Mittelwert, oberes/unteres Band), auf denen die Entry-/Exit-Signale tatsaechlich
ausgeloest wurden.

Marker:
- **Rot** = Entry (`entry_time`), **Gruen** = Exit (`exit_time`).
- Exit-Grund im Subplot-Titel: `signal` (Rueckkehr zum Mittelwert), `holding_limit`
  (Zeitlimit erreicht) oder `day_end` (Ende des verfuegbaren Preisfensters).

Muss NACH Abschnitt 6b ausgefuehrt werden (braucht `bo_last_result["test_trades_df"]`
und `bo_last_result["best_params"]`).

In [16]:
# =====================================================================
# Widgets: Tag-Navigation ueber die discovery_date-Werte in bo_test_trades_df
# =====================================================================
be_w_day_label = widgets.HTML()
be_w_prev_button = widgets.Button(description="< Tag", layout=widgets.Layout(width="90px"))
be_w_next_button = widgets.Button(description="Tag >", layout=widgets.Layout(width="90px"))
be_output = widgets.Output()
be_state = {"day_idx": 0}


def _be_current_days():
    """Sortierte, eindeutige discovery_date-Werte MIT mindestens einem Trade
    in bo_last_result["test_trades_df"] (siehe Abschnitt 6b) -- Tage ohne
    Trade werden uebersprungen, da es dort nichts zu markieren gaebe."""
    if not bo_last_result or "test_trades_df" not in bo_last_result:
        return pd.DatetimeIndex([])
    trades = bo_last_result["test_trades_df"]
    if trades.empty:
        return pd.DatetimeIndex([])
    return pd.DatetimeIndex(sorted(trades["discovery_date"].unique()))


def _be_render(_=None):
    with be_output:
        clear_output(wait=True)

        if not bo_last_result or "test_trades_df" not in bo_last_result:
            print("Erst Abschnitt 6b (finale Testzeitraum-Auswertung) ausfuehren.")
            return

        trades = bo_last_result["test_trades_df"]
        if trades.empty:
            print("test_trades_df ist leer -- keine Trades im finalen Testzeitraum.")
            return

        days = _be_current_days()
        be_state["day_idx"] = max(0, min(be_state["day_idx"], len(days) - 1))
        day = days[be_state["day_idx"]]
        be_w_day_label.value = f"<b>{day.date()}</b> &nbsp; (Tag {be_state['day_idx'] + 1} von {len(days)} mit Trade(s))"

        day_trades = trades.loc[trades["discovery_date"] == day]

        basket = bo_last_result["result"].basket
        best_params = bo_last_result["best_params"]
        cliques_df_day = bo_last_result["cliques_df"]
        clique_row = cliques_df_day.loc[cliques_df_day["discovery_date"] == day]
        if clique_row.empty:
            print(f"Keine Clique-Definition fuer {day.date()} gefunden (Basket {basket}).")
            return
        asset_ids = [a for a in clique_row.iloc[0]["asset_ids"] if a in ohlcv_df[ASSET_ID_COL].unique()]
        symbols = [id_to_symbol.get(a, f"ID:{a}") for a in asset_ids]
        if len(symbols) < 2:
            print(f"Clique an {day.date()} hat weniger als 2 handelbare Assets.")
            return

        print(f"Clique-Mitglieder ({len(symbols)}): {', '.join(symbols)} | "
              f"{len(day_trades)} Trade(s) an diesem Tag")

        day_start = day.normalize()
        day_end = day_start + pd.Timedelta(days=1)
        price_matrix_day = ohlcv_df.pivot_table(
            index=TIMESTAMP_COL, columns=ASSET_ID_COL, values=CLOSE_COL, aggfunc="last",
        ).sort_index()

        # Bollinger-Baender brauchen Warmup-Vorlauf vor dem Tag (lookback_candles,
        # analog run_bollinger_backtest) fuer den rollierenden Mittelwert/Std --
        # gleiche Konvention wie Abschnitt 5c (dort: lookback_candles + 30).
        lookback_candles = best_params["lookback_candles"]
        hedge_lookback_candles = best_params["hedge_lookback_candles"]
        exit_lookback_candles = best_params["exit_lookback_candles"]
        warmup_candles = max(lookback_candles, hedge_lookback_candles, exit_lookback_candles) + 30
        ts_index = price_matrix_day.index
        day_start_pos = ts_index.searchsorted(day_start)
        day_end_pos = ts_index.searchsorted(day_end)
        window_start_pos = max(0, day_start_pos - warmup_candles)

        if day_start_pos - warmup_candles < 0:
            print(f"Zu wenig Historie vor {day.date()} fuer Warmup ({warmup_candles} Kerzen) -- uebersprungen.")
            return

        window_prices = price_matrix_day.iloc[window_start_pos:day_end_pos][asset_ids]
        window_prices = window_prices.dropna(how="any")
        if len(window_prices) < warmup_candles + 10:
            print(f"Nur {len(window_prices)} vollstaendige Kerzen im Warmup-Fenster -- zu wenig fuer eine sinnvolle Darstellung.")
            return

        spread_df_day, _hedge_ratios_day = bpe.build_pairwise_price_spreads(
            window_prices, hedge_lookback_candles=hedge_lookback_candles,
        )
        bands_day = bpe.compute_bollinger_bands(
            spread_df_day, lookback_candles=lookback_candles, n_std=best_params["n_std"],
        )
        exit_mean_day = bpe.compute_exit_mean(
            spread_df_day, exit_lookback_candles=exit_lookback_candles,
        )

        day_slice_day = spread_df_day.index[spread_df_day.index >= day_start]

        import matplotlib.pyplot as plt
        from itertools import combinations
        from matplotlib.lines import Line2D

        pairs = list(combinations(asset_ids, 2))
        n_pairs = len(pairs)
        n_cols = min(2, n_pairs)
        n_rows = int(np.ceil(n_pairs / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(8.5 * n_cols, 5.5 * n_rows), squeeze=False)

        for i, pair in enumerate(pairs):
            ax = axes[i // n_cols][i % n_cols]
            t1_sym = id_to_symbol.get(pair[0], f"ID:{pair[0]}")
            t2_sym = id_to_symbol.get(pair[1], f"ID:{pair[1]}")

            if pair not in bands_day:
                ax.axis("off")
                continue
            band_day = bands_day[pair].loc[day_slice_day]
            exit_mean_pair_day = exit_mean_day[pair].loc[day_slice_day]

            ax.plot(band_day.index, band_day["spread"], color="#2a2a28", linewidth=1.1, label="Preis-Spread")
            ax.plot(band_day.index, band_day["mid"], color="#898781", linewidth=1, linestyle="--", label="Gleitender Mittelwert")
            ax.plot(band_day.index, band_day["upper"], color="#c1443f", linewidth=0.8, linestyle=":", label="Oberes Band")
            ax.plot(band_day.index, band_day["lower"], color="#256abf", linewidth=0.8, linestyle=":", label="Unteres Band")
            ax.plot(exit_mean_pair_day.index, exit_mean_pair_day.values, color="#1baf7a", linewidth=1.2, linestyle="-.", label="Exit-Mittelwert")

            pair_syms = {t1_sym, t2_sym}
            pair_trades = day_trades.loc[
                day_trades.apply(lambda r: {r["asset_long_sym"], r["asset_short_sym"]} == pair_syms, axis=1)
            ]
            exit_reasons = []
            for _, trade in pair_trades.iterrows():
                if trade["entry_time"] in band_day.index:
                    ax.scatter([trade["entry_time"]], [band_day["spread"].loc[trade["entry_time"]]], color="firebrick", zorder=5, s=35, alpha=0.75)
                if trade["exit_time"] in band_day.index:
                    ax.scatter([trade["exit_time"]], [band_day["spread"].loc[trade["exit_time"]]], color="seagreen", zorder=5, s=35, marker="s", alpha=0.75)
                exit_reasons.append(trade["exit_reason"])

            title = f"{t1_sym}-{t2_sym}"
            if exit_reasons:
                reason_counts = pd.Series(exit_reasons).value_counts()
                reason_str = ", ".join(f"{r}×{c}" for r, c in reason_counts.items())
                title += f" ({len(exit_reasons)} Trade(s), Exit: {reason_str})"
            ax.set_title(title, fontsize=9)
            ax.tick_params(axis="x", rotation=45, labelsize=7)
            ax.tick_params(axis="y", labelsize=7)

        for j in range(n_pairs, n_rows * n_cols):
            axes[j // n_cols][j % n_cols].axis("off")

        legend_handles = [
            Line2D([0], [0], color="firebrick", marker="o", linestyle="-", markersize=5, label="Entry"),
            Line2D([0], [0], color="seagreen", marker="s", linestyle="-", markersize=5, label="Exit"),
            Line2D([0], [0], color="#898781", linestyle="--", linewidth=1.2, label="Gleitender Mittelwert"),
            Line2D([0], [0], color="#c1443f", linestyle=":", linewidth=1.2, label="Bollinger-Band (+/- n_std)"),
            Line2D([0], [0], color="#1baf7a", linestyle="-.", linewidth=1.2, label="Exit-Mittelwert"),
        ]
        fig.legend(handles=legend_handles, loc="upper right", fontsize=8)
        fig.suptitle(f"Trades -- {basket} (Bollinger) -- {day.date()}")
        plt.tight_layout()
        plt.show()

        display(day_trades[[
            "pair", "asset_long_sym", "asset_short_sym", "direction", "entry_time", "exit_time",
            "exit_reason", "gross_return", "net_return", "holding_candles",
        ]].reset_index(drop=True))


def _be_on_prev(_button=None):
    be_state["day_idx"] -= 1
    _be_render()


def _be_on_next(_button=None):
    be_state["day_idx"] += 1
    _be_render()


be_w_prev_button.on_click(_be_on_prev)
be_w_next_button.on_click(_be_on_next)

be_ui = widgets.VBox([
    widgets.HBox([be_w_prev_button, be_w_day_label, be_w_next_button]),
    be_output,
])
display(be_ui)
_be_render()

# 7. GARCH-Toleranzintervall-Variante (Renditedifferenz, GARCH(1,1)-Vola, Exit-Mitte)

Vierte, PARALLELE Strategie neben Abschnitt 5 (EWMA-Toleranzintervall), Abschnitt 5d
(schneller Tuner) und Abschnitt 6 (Bollinger-Baender). Nutzt denselben Cliquen-Cache
und dieselben OHLCV-Daten, aber eine GARCH(1,1)-basierte Signalkette
(`src/signals/garch.py`, `src/signals/garch_tolerance.py`, `garch_pairtrading_engine.py`):

1. **Renditedifferenz-Spread** wie in Abschnitt 5 (Lin, Chen & Syu 2021): `Rt = r_Ai,t - r_Aj,t`.
2. **GARCH(1,1)-Volatilitaet statt EWMA**: `sigma_t` wird PRO PAAR, PRO CLIQUE-TAG per
   Maximum-Likelihood gefittet (Student-t-Innovationen, siehe Aufgabenstellung-Zusatz-PDF:
   "Student-t GARCH besser fuer Crypto-Fat-Tails als Standard-GARCH") -- kein fester
   decay-Parameter wie bei EWMA, sondern datengetriebene `omega/alpha/beta/nu`. Eigene
   Implementierung (`src/signals/garch.py`) statt des `arch`-Pakets: dessen interner
   Optimierer stuerzt auf diesem Windows-Setup mit einem BLAS/DLL-Konflikt ab
   (reproduzierbar bereits bei einem nackten `scipy.optimize.minimize(..., method="BFGS")`),
   `method="Nelder-Mead"` ist nachweislich stabil.
3. **Xmax_t** + **Wilks-Toleranzintervall** wie in Abschnitt 5 -- Entry bei Grenzdurchbruch.
4. **Exit-Mitte** (Aufgabenstellung-Zusatz): der Trade schliesst, sobald `|Xt|` DES
   GEHANDELTEN PAARES die MITTE zwischen unterer und oberer Toleranzgrenze erreicht/
   unterschreitet -- `exit_mid = (lower_limit + upper_limit) / 2`. Kein separates
   Lookback-Fenster noetig (anders als Abschnitt 5/6), die Referenzlinie ergibt sich
   direkt aus den ohnehin fuer den Entry berechneten Grenzen.

**Rechenkosten:** GARCH-Fits sind pro Paar/Tag deutlich teurer als EWMA -- kleinere
`n_trials`/Walk-Forward-Fenster-Anzahl fuer die erste Exploration empfohlen (Default hier: 20
statt 50 Trials).

**Wie in Abschnitt 5/6**: eigener Optuna-Tuner (`tune_garch.py`, Walk-Forward-Backtest),
eigene persistente SQLite-Study (`garch_study.db`, getrennt von den anderen drei
Storage-Dateien), damit die vier Strategien direkt via
`pairtrading_engine.plot_strategy_comparison` gegenuebergestellt werden koennen, ohne
sich gegenseitig im selben Storage zu vermischen.

In [17]:
# =====================================================================
# Abschnitt 7: GARCH-Tuning -- Widgets
# =====================================================================
import garch_pairtrading_engine as ge
import tune_garch as tgc

gc_w_variant = widgets.Dropdown(options=["raw", "residual"], value="raw", description="Cliquen-Variante:", style={"description_width": "160px"})
gc_w_basket = widgets.Dropdown(options=basket_labels_available, value=basket_labels_available[0], description="Basket:", style={"description_width": "160px"})

gc_w_overall_start = widgets.DatePicker(description="Gesamt-Start:", style={"description_width": "160px"})
gc_w_overall_end = widgets.DatePicker(description="Gesamt-Ende:", style={"description_width": "160px"})
gc_w_test_pct = widgets.IntSlider(value=15, min=5, max=50, description="Testanteil (%):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
gc_w_embargo_days = widgets.IntSlider(value=5, min=0, max=30, description="Embargo (Tage):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))

gc_w_train_val_start = widgets.DatePicker(description="-> Train/Val Start:", disabled=True, style={"description_width": "160px"})
gc_w_train_val_end = widgets.DatePicker(description="-> Train/Val Ende:", disabled=True, style={"description_width": "160px"})
gc_w_test_start = widgets.DatePicker(description="-> Test Start:", disabled=True, style={"description_width": "160px"})
gc_w_test_end = widgets.DatePicker(description="-> Test Ende:", disabled=True, style={"description_width": "160px"})


def _gc_default_overall_range():
    """Wie _ot_default_overall_range (Abschnitt 5). Bewusst KURZ gehalten
    (2 Wochen statt 2 Monate) -- GARCH-Fits sind teuer, siehe Abschnitt-7-Intro."""
    fixed_start = pd.Timestamp("2025-01-01")
    fixed_end = pd.Timestamp("2025-01-15")
    data_min_ts = pd.Timestamp(data_min)
    data_max_ts = pd.Timestamp(data_max)
    if fixed_start >= data_min_ts and fixed_end <= data_max_ts:
        return fixed_start.date(), fixed_end.date()
    return data_min, data_max


_gc_os, _gc_oe = _gc_default_overall_range()
gc_w_overall_start.value = _gc_os
gc_w_overall_end.value = _gc_oe


def _gc_recompute_split(change=None):
    """Wie _bo_recompute_split (Abschnitt 6): Train/Val + Embargo + Test aus
    Gesamtzeitraum und Testanteil berechnen."""
    if gc_w_overall_start.value is None or gc_w_overall_end.value is None:
        return
    overall_start = pd.Timestamp(gc_w_overall_start.value)
    overall_end = pd.Timestamp(gc_w_overall_end.value)
    if overall_end <= overall_start:
        return

    total_days = (overall_end - overall_start).days
    test_days = max(1, int(round(total_days * gc_w_test_pct.value / 100)))
    embargo_days = gc_w_embargo_days.value
    test_end = overall_end

    train_val_start = overall_start
    train_val_end = overall_end - pd.Timedelta(days=test_days + embargo_days)
    if train_val_end <= train_val_start:
        train_val_end = train_val_start + pd.Timedelta(days=1)
    test_start = train_val_end + pd.Timedelta(days=embargo_days)
    if test_start > test_end:
        test_start = test_end

    gc_w_train_val_start.value = train_val_start.date()
    gc_w_train_val_end.value = train_val_end.date()
    gc_w_test_start.value = test_start.date()
    gc_w_test_end.value = test_end.date()


gc_w_overall_start.observe(_gc_recompute_split, names="value")
gc_w_overall_end.observe(_gc_recompute_split, names="value")
gc_w_test_pct.observe(_gc_recompute_split, names="value")
gc_w_embargo_days.observe(_gc_recompute_split, names="value")
_gc_recompute_split()

gc_w_train_days = widgets.IntSlider(value=7, min=2, max=30, description="Trainingsfenster (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
gc_w_val_days = widgets.IntSlider(value=3, min=1, max=15, description="Validierungsfenster (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
gc_w_step_days = widgets.IntSlider(value=3, min=1, max=30, description="Schrittweite (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
gc_w_wf_embargo_days = widgets.IntSlider(value=1, min=0, max=10, description="WF-internes Embargo (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))

gc_w_n_trials = widgets.IntSlider(value=20, min=2, max=100, step=2, description="n_trials (NEUE Trials):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
gc_w_seed = widgets.IntText(value=42, description="Seed:", style={"description_width": "160px"})

gc_w_persistent = widgets.Checkbox(value=True, description="Persistenter SQLite-Storage (fuer optuna-dashboard)", indent=False)
gc_w_study_name = widgets.Text(value="", placeholder="Default: <Basket>_garch", description="Study-Name:", style={"description_width": "160px"})
gc_w_storage_path = widgets.Text(value=str(tgc.DEFAULT_STORAGE_PATH), description="SQLite-Datei:", style={"description_width": "160px"}, layout=widgets.Layout(width="560px"))

gc_run_button = widgets.Button(description="GARCH-Tuning starten", button_style="warning", icon="play")
gc_output = widgets.Output()
gc_last_result = {}


def _gc_row(widget, description):
    desc_html = widgets.HTML(
        f"<span style='color:#888; font-size:0.9em;'>{description}</span>",
        layout=widgets.Layout(margin="6px 0 0 12px"),
    )
    return widgets.HBox([widget, desc_html])


gc_ui = widgets.VBox([
    widgets.HTML("<b>Universum</b>"),
    _gc_row(gc_w_variant, "Cliquen-Cache-Variante (siehe Abschnitt 2): 'raw' oder 'residual'."),
    _gc_row(gc_w_basket, "Welcher TDA-Basket (Cluster-Gruppe) getunt wird."),

    widgets.HTML("<b>Zeitliche Trennung</b> (EIN Gesamtzeitraum -- Train/Val/Test werden automatisch berechnet, bewusst kurz wegen GARCH-Kosten)"),
    _gc_row(widgets.HBox([gc_w_overall_start, gc_w_overall_end]), "Gesamter Zeitraum, der geladen/verwendet wird."),
    _gc_row(gc_w_test_pct, "Wie viel Prozent des Gesamtzeitraums (am ENDE) als finaler, unberuehrter Testzeitraum reserviert werden."),
    _gc_row(gc_w_embargo_days, "Puffer (Tage) zwischen Train/Val Ende und Test Start."),
    widgets.HTML("<span style='color:#aaa; font-size:0.85em;'>Automatisch berechnet (nicht editierbar):</span>"),
    _gc_row(widgets.HBox([gc_w_train_val_start, gc_w_train_val_end]), "Zeitraum, INNERHALB dessen Optuna sucht (Walk-Forward-Fenster darin)."),
    _gc_row(widgets.HBox([gc_w_test_start, gc_w_test_end]), "Bleibt waehrend des gesamten Tunings unberuehrt, wird erst unten (Abschnitt 7b) verwendet."),

    widgets.HTML("<b>Walk-Forward-Fenster (innerhalb Train/Val)</b>"),
    _gc_row(gc_w_train_days, "Laenge des Trainingsfensters je Walk-Forward-Schritt (Tage)."),
    _gc_row(gc_w_val_days, "Laenge des Validierungsfensters je Walk-Forward-Schritt (Tage)."),
    _gc_row(gc_w_step_days, "Um wie viele Tage das Fenster bei jedem Walk-Forward-Schritt weiterrueckt."),
    _gc_row(gc_w_wf_embargo_days, "Puffer (Tage) zwischen Training und Validierung INNERHALB eines Walk-Forward-Schritts."),

    widgets.HTML("<b>Optuna</b>"),
    _gc_row(gc_w_n_trials, "Anzahl NEUER Trials in diesem Lauf -- bewusst niedriger Default wegen GARCH-Fit-Kosten pro Trial."),
    _gc_row(gc_w_seed, "Zufalls-Seed fuer den TPE-Sampler -- nur bei einer NEU erstellten Study wirksam."),

    widgets.HTML("<b>Persistenz & Dashboard</b>"),
    _gc_row(gc_w_persistent, "An: eigene SQLite-Datei (getrennt von Abschnitt 5/6), live mit optuna-dashboard verfolgbar."),
    _gc_row(gc_w_study_name, "Name der Optuna-Study. Leer = '<Basket>_garch'."),
    _gc_row(gc_w_storage_path, "Pfad zur SQLite-Datei, in der die Study gespeichert wird."),

    gc_run_button,
    gc_output,
])
display(gc_ui)

In [18]:
# =====================================================================
# Abschnitt 7: Callback -- startet einen GARCH-Tuning-Lauf ueber
# tune_garch.tune_garch_strategy() und zeigt die Ergebnisse an.
# =====================================================================
import matplotlib.pyplot as plt


def _gc_on_run_clicked(_button=None):
    with gc_output:
        clear_output(wait=True)

        for w, label in [
            (gc_w_train_val_start, "Train/Val Start"), (gc_w_train_val_end, "Train/Val Ende"),
            (gc_w_test_start, "Test Start"), (gc_w_test_end, "Test Ende"),
        ]:
            if w.value is None:
                print(f"Bitte '{label}' setzen.")
                return

        cliques_df_gc = cliques_by_variant[gc_w_variant.value]
        cliques_df_gc = cliques_df_gc.loc[cliques_df_gc["basket"] == gc_w_basket.value]

        effective_study_name = gc_w_study_name.value or f"{gc_w_basket.value}_garch"
        print(f"GARCH-Tuning laeuft -- Basket: {gc_w_basket.value} | "
              f"Train/Val: {gc_w_train_val_start.value} -> {gc_w_train_val_end.value} | "
              f"Test (unberuehrt bis Abschnitt 7b): {gc_w_test_start.value} -> {gc_w_test_end.value} | "
              f"n_trials={gc_w_n_trials.value} (GARCH-Fits sind teuer -- das kann eine Weile dauern) ...")
        if gc_w_persistent.value:
            print(f"Persistenter Storage: sqlite:///{gc_w_storage_path.value} | Study-Name: '{effective_study_name}'\n")
        else:
            print("Persistenter Storage: AUS (In-Memory, kein Dashboard).\n")

        try:
            result = tgc.tune_garch_strategy(
                basket=gc_w_basket.value, cliques_df=cliques_df_gc, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
                timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
                train_val_start=gc_w_train_val_start.value, train_val_end=gc_w_train_val_end.value,
                test_start=gc_w_test_start.value, test_end=gc_w_test_end.value,
                train_days=gc_w_train_days.value, val_days=gc_w_val_days.value,
                step_days=gc_w_step_days.value, embargo_days=gc_w_wf_embargo_days.value,
                n_trials=gc_w_n_trials.value, seed=gc_w_seed.value,
                output_path=PROCESSED_DIR / "tuning" / f"garch_trials_{gc_w_basket.value.replace(' ', '_')}.parquet",
                study_name=gc_w_study_name.value or None,
                storage_path=gc_w_storage_path.value,
                use_persistent_storage=gc_w_persistent.value,
            )
        except (ValueError, RuntimeError) as e:
            print(f"Tuning abgebrochen: {e}")
            return

        gc_last_result.clear()
        gc_last_result.update(
            result=result, cliques_df=cliques_df_gc,
            test_start=gc_w_test_start.value, test_end=gc_w_test_end.value,
        )

        tgc.summarize_garch_tuning_results([result])

        completed = result.trials_df.loc[result.trials_df["status"] == "completed"].sort_values("objective", ascending=False)
        if not completed.empty:
            print("\nTop 10 abgeschlossene Trials nach Objective:")
            display(completed.head(10)[[
                "lookback_candles", "p_content", "trade_mode", "recalib_every",
                "max_holding_hours", "max_concurrent_positions",
                "objective", "mean_fold_return", "std_fold_return",
            ]])

            fig, ax = plt.subplots(figsize=(7, 4.5))
            ax.scatter(completed["trial_number"], completed["objective"], color="steelblue")
            ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
            ax.set_xlabel("Trial-Nummer")
            ax.set_ylabel("Objective (mittlere Fold-Rendite)")
            ax.set_title(f"GARCH-Trials: Objective ueber die Zeit ({gc_w_basket.value})")
            plt.tight_layout()
            plt.show()


gc_run_button.on_click(_gc_on_run_clicked)

## 7b. Finale Testzeitraum-Auswertung (GARCH-Variante)

Wie Abschnitt 5b/6b, aber fuer die GARCH-Toleranzintervall-Strategie: laeuft
`run_garch_backtest` mit den `best_params` aus Abschnitt 7 GENAU EINMAL auf dem finalen
Testzeitraum, der waehrend des gesamten Tunings nie angefasst wurde.

In [19]:
# =====================================================================
# Abschnitt 7b: EINMALIGE finale Auswertung auf dem unberuehrten Testzeitraum
# =====================================================================
if not gc_last_result:
    print("Erst Abschnitt 7 (GARCH-Tuning) ausfuehren.")
else:
    gc_result = gc_last_result["result"]
    gc_best_params = gc_result.best_params

    print(f"Finale Testzeitraum-Auswertung (GARCH) -- Basket: {gc_result.basket} | "
          f"Testzeitraum: {gc_last_result['test_start']} -> {gc_last_result['test_end']} | "
          f"best_params={gc_best_params}\n")

    gc_max_holding_candles_test = max(1, int(round(gc_best_params["max_holding_hours"] * 60 / 5)))

    gc_test_trades_df = ge.run_garch_backtest(
        cliques_df=gc_last_result["cliques_df"], ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
        timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
        start_date=gc_last_result["test_start"], end_date=gc_last_result["test_end"],
        lookback_candles=gc_best_params["lookback_candles"], p_content=gc_best_params["p_content"],
        confidence=tgc.FIXED_CONFIDENCE, max_holding_candles=gc_max_holding_candles_test,
        fee_bps=tgc.FIXED_FEE_BPS, trade_mode=gc_best_params["trade_mode"],
        recalib_every=gc_best_params["recalib_every"],
        garch_dist=tgc.FIXED_GARCH_DIST, garch_max_iter=tgc.FIXED_GARCH_MAX_ITER,
    )
    gc_test_equity_curve = pte.compute_equity_curve(
        gc_test_trades_df, initial_capital=tgc.FIXED_INITIAL_CAPITAL,
        max_concurrent_positions=gc_best_params["max_concurrent_positions"],
    )
    gc_test_stats = pte.compute_backtest_stats(gc_test_trades_df, gc_test_equity_curve)

    gc_last_result["test_trades_df"] = gc_test_trades_df
    gc_last_result["best_params"] = gc_best_params

    if gc_test_trades_df.empty:
        print("Keine Trades im finalen Testzeitraum mit den gewaehlten Parametern.")
    else:
        gc_test_stats_display = pd.DataFrame([{
            "Anzahl Trades": gc_test_stats["n_trades"],
            "Win-Rate": f"{gc_test_stats['win_rate']:.1%}",
            "Sharpe": f"{gc_test_stats['sharpe']:.2f}" if np.isfinite(gc_test_stats["sharpe"]) else "n/a",
            "Max Drawdown": f"{gc_test_stats['max_drawdown']:.2%}" if np.isfinite(gc_test_stats["max_drawdown"]) else "n/a",
            "Gesamtrendite (finaler Test, out-of-sample)": f"{gc_test_stats['total_return']:.2%}" if np.isfinite(gc_test_stats["total_return"]) else "n/a",
            "Durchschnittliche Haltedauer": f"{gc_test_stats['avg_holding_hours']:.1f}h" if np.isfinite(gc_test_stats["avg_holding_hours"]) else "n/a",
        }]).T
        gc_test_stats_display.columns = ["Wert"]
        display(gc_test_stats_display)
        pte.plot_equity_curve(gc_test_equity_curve, initial_capital=tgc.FIXED_INITIAL_CAPITAL)

        display(gc_test_trades_df[[
            "pair", "asset_long_sym", "asset_short_sym", "direction", "entry_time", "exit_time",
            "exit_reason", "gross_return", "net_return", "holding_candles",
        ]].reset_index(drop=True))

Erst Abschnitt 7 (GARCH-Tuning) ausfuehren.


## 7c. Trade-Explorer: Xmax mit Toleranzgrenzen und Exit-Mitte (finaler Testzeitraum)

Wie Abschnitt 5c/6c, aber fuer die GARCH-Strategie: zeigt die **tatsaechlichen Trades**
aus Abschnitt 7b (`test_trades_df`, mit `best_params` auf dem unberuehrten Testzeitraum
simuliert) tageweise als Xmax-Plots MIT den Wilks-Toleranzgrenzen UND der Exit-Mitte
(`(lower_limit + upper_limit) / 2`), auf denen die Entry-/Exit-Signale tatsaechlich
ausgeloest wurden (`garch_pairtrading_engine.plot_xmax_with_garch_bands`).

Marker:
- **Farbige Punkte** = Entry je Paar (siehe Legende).
- Exit-Grund im Subplot-Titel: `signal` (Xt erreicht Exit-Mitte), `holding_limit`
  (Zeitlimit erreicht) oder `day_end` (Ende des verfuegbaren Preisfensters).

Muss NACH Abschnitt 7b ausgefuehrt werden (braucht `gc_last_result["test_trades_df"]`
und `gc_last_result["best_params"]`).

In [20]:
# =====================================================================
# Widgets: Tag-Navigation ueber die discovery_date-Werte in gc_test_trades_df
# =====================================================================
ge_w_day_label = widgets.HTML()
ge_w_prev_button = widgets.Button(description="< Tag", layout=widgets.Layout(width="90px"))
ge_w_next_button = widgets.Button(description="Tag >", layout=widgets.Layout(width="90px"))
ge_output = widgets.Output()
ge_state = {"day_idx": 0}


def _ge_current_days():
    """Sortierte, eindeutige discovery_date-Werte MIT mindestens einem Trade
    in gc_last_result["test_trades_df"] (siehe Abschnitt 7b) -- Tage ohne
    Trade werden uebersprungen, da es dort nichts zu markieren gaebe."""
    if not gc_last_result or "test_trades_df" not in gc_last_result:
        return pd.DatetimeIndex([])
    trades = gc_last_result["test_trades_df"]
    if trades.empty:
        return pd.DatetimeIndex([])
    return pd.DatetimeIndex(sorted(trades["discovery_date"].unique()))


def _ge_render(_=None):
    with ge_output:
        clear_output(wait=True)

        if not gc_last_result or "test_trades_df" not in gc_last_result:
            print("Erst Abschnitt 7b (finale Testzeitraum-Auswertung) ausfuehren.")
            return

        trades = gc_last_result["test_trades_df"]
        if trades.empty:
            print("test_trades_df ist leer -- keine Trades im finalen Testzeitraum.")
            return

        days = _ge_current_days()
        ge_state["day_idx"] = max(0, min(ge_state["day_idx"], len(days) - 1))
        day = days[ge_state["day_idx"]]
        ge_w_day_label.value = f"<b>{day.date()}</b> &nbsp; (Tag {ge_state['day_idx'] + 1} von {len(days)} mit Trade(s))"

        day_trades = trades.loc[trades["discovery_date"] == day]

        basket = gc_last_result["result"].basket
        best_params = gc_last_result["best_params"]
        cliques_df_day = gc_last_result["cliques_df"]
        clique_row = cliques_df_day.loc[cliques_df_day["discovery_date"] == day]
        if clique_row.empty:
            print(f"Keine Clique-Definition fuer {day.date()} gefunden (Basket {basket}).")
            return
        asset_ids = [a for a in clique_row.iloc[0]["asset_ids"] if a in ohlcv_df[ASSET_ID_COL].unique()]
        symbols = [id_to_symbol.get(a, f"ID:{a}") for a in asset_ids]
        if len(symbols) < 2:
            print(f"Clique an {day.date()} hat weniger als 2 handelbare Assets.")
            return

        print(f"Clique-Mitglieder ({len(symbols)}): {', '.join(symbols)} | "
              f"{len(day_trades)} Trade(s) an diesem Tag")

        day_start = day.normalize()
        day_end = day_start + pd.Timedelta(days=1)
        price_matrix_day = ohlcv_df.pivot_table(
            index=TIMESTAMP_COL, columns=ASSET_ID_COL, values=CLOSE_COL, aggfunc="last",
        ).sort_index()

        lookback_candles = best_params["lookback_candles"]
        warmup_candles = lookback_candles + 30
        ts_index = price_matrix_day.index
        day_start_pos = ts_index.searchsorted(day_start)
        day_end_pos = ts_index.searchsorted(day_end)
        window_start_pos = max(0, day_start_pos - warmup_candles)

        if day_start_pos - warmup_candles < 0:
            print(f"Zu wenig Historie vor {day.date()} fuer Warmup ({warmup_candles} Kerzen) -- uebersprungen.")
            return

        window_prices = price_matrix_day.iloc[window_start_pos:day_end_pos][asset_ids]
        window_prices = window_prices.dropna(how="any")
        if len(window_prices) < warmup_candles + 10:
            print(f"Nur {len(window_prices)} vollstaendige Kerzen im Warmup-Fenster -- zu wenig fuer eine sinnvolle Darstellung.")
            return

        spread_df_day = build_pairwise_return_spreads(window_prices)
        adjusted_df_day, _fit_info_day = compute_garch_adjusted_spreads(spread_df_day)
        max_spread_df_day = compute_max_adjusted_spread(adjusted_df_day)
        tolerance_df_day = generate_tolerance_signals(
            max_spread_df_day, lookback_candles=lookback_candles,
            p_content=best_params["p_content"], confidence=tgc.FIXED_CONFIDENCE,
            recalib_every=best_params["recalib_every"],
        )
        day_slice_day = max_spread_df_day.index[max_spread_df_day.index >= day_start]
        max_spread_df_day = max_spread_df_day.loc[day_slice_day]
        tolerance_df_day = tolerance_df_day.loc[day_slice_day]

        print(f"GARCH-Fit lief pro Paar auf diesem Warmup-Fenster ({len(window_prices)} Kerzen) -- "
              f"kann einige Sekunden dauern.")

        ge.plot_xmax_with_garch_bands(max_spread_df_day, tolerance_df_day, day_trades)

        display(day_trades[[
            "pair", "asset_long_sym", "asset_short_sym", "direction", "entry_time", "exit_time",
            "exit_reason", "gross_return", "net_return", "holding_candles",
        ]].reset_index(drop=True))


def _ge_on_prev(_button=None):
    ge_state["day_idx"] -= 1
    _ge_render()


def _ge_on_next(_button=None):
    ge_state["day_idx"] += 1
    _ge_render()


ge_w_prev_button.on_click(_ge_on_prev)
ge_w_next_button.on_click(_ge_on_next)

ge_ui = widgets.VBox([
    widgets.HBox([ge_w_prev_button, ge_w_day_label, ge_w_next_button]),
    ge_output,
])
display(ge_ui)
_ge_render()

# 8. Ergebnis-Browser (aus Cache, ohne erneutes Tuning/Backtesting)

Abschnitte 5b/6b/7b (finale Testauswertung) und 5c/6c/7c (Trade-Explorer) rechnen
bei JEDEM Ausfuehren live: Backtest neu simulieren, und im Explorer pro Tag die
Signalserien (EWMA/Bollinger/GARCH) neu aus den OHLCV-Rohdaten berechnen -- bei
GARCH kostet das pro Tag mehrere Sekunden (Maximum-Likelihood-Fit pro Paar).

Dieser Abschnitt trennt **Berechnen** von **Ansehen**:

- **8a. Cache schreiben** -- EINMAL pro Strategie/Basket ausfuehren, NACHDEM die
  jeweilige Testauswertung (5b/6b/7b) bereits gelaufen ist. Iteriert ueber alle
  Handelstage mit Trades und speichert die Trades, die Equity-Kurve und die
  Signalserien jedes Tages unter `data/processed/results_cache/` (siehe
  `Src.results_cache`-Modul-Docstring fuer das Dateilayout).
- **8b. Ergebnis-Browser** -- laedt AUSSCHLIESSLICH aus diesem Cache (Dropdown:
  Strategie + Basket, dann Tag-Navigation wie in 5c/6c/7c). Kein OHLCV-Zugriff,
  kein GARCH-Fit, kein Tuning-Objekt (`ot_last_result`/`bo_last_result`/
  `gc_last_result`) noetig -- funktioniert auch in einem frischen Kernel oder
  nachdem die In-Memory-Tuning-Ergebnisse laengst weg sind.

**Wichtig:** 8a liest weiterhin von `ot_last_result`/`bo_last_result`/
`gc_last_result` (den bereits im Speicher stehenden Testauswertungen aus 5b/6b/7b)
-- 8a selbst fuehrt KEIN neues Tuning durch, sondern schreibt nur, was bereits
berechnet wurde, zusaetzlich auf Disk. Erst NACH einem erfolgreichen 8a-Lauf
funktioniert 8b unabhaengig davon.

## 8a. Cache schreiben (einmal pro Strategie/Basket, nach 5b/6b/7b)

In [21]:
# =====================================================================
# Abschnitt 8a: Testauswertung + Tages-Signalserien in results_cache/ ablegen
# =====================================================================
from Src.results_cache import save_test_result, save_day_signals

def _cache_ewma_result():
    """Cached Abschnitt 5b/5c (EWMA-Toleranzintervall-Variante)."""
    if not ot_last_result or "test_trades_df" not in ot_last_result:
        print("Erst Abschnitt 5b ausfuehren -- nichts zu cachen.")
        return

    trades = ot_last_result["test_trades_df"]
    result = ot_last_result["result"]
    eff = ot_last_result["effective_params"]
    basket = result.basket

    equity_curve = pte.compute_equity_curve(
        trades, initial_capital=tb.FIXED_INITIAL_CAPITAL,
        max_concurrent_positions=eff["max_concurrent_positions"],
    )
    stats = pte.compute_backtest_stats(trades, equity_curve)
    save_test_result(
        variant="ewma", basket=basket,
        test_start=ot_w_test_start.value, test_end=ot_w_test_end.value,
        params=eff, stats=stats, trades_df=trades, equity_curve=equity_curve,
    )

    if trades.empty:
        print(f"EWMA/{basket}: keine Trades -- Cache nur mit leeren Tabellen geschrieben.")
        return

    price_matrix = ohlcv_df.pivot_table(
        index=TIMESTAMP_COL, columns=ASSET_ID_COL, values=CLOSE_COL, aggfunc="last",
    ).sort_index()
    cliques_df_day = ot_last_result["cliques_df_test"]
    lookback_candles = eff["lookback_candles"]
    exit_lookback_candles = eff["exit_lookback_candles"]
    hedge_lookback_candles = eff.get("hedge_lookback_candles", exit_lookback_candles)
    warmup_candles = max(lookback_candles, exit_lookback_candles) + 30

    days = pd.DatetimeIndex(sorted(trades["discovery_date"].unique()))
    n_cached = 0
    for day in days:
        day_start = day.normalize()
        day_end = day_start + pd.Timedelta(days=1)
        clique_row = cliques_df_day.loc[cliques_df_day["discovery_date"] == day]
        if clique_row.empty:
            continue
        asset_ids = [a for a in clique_row.iloc[0]["asset_ids"] if a in ohlcv_df[ASSET_ID_COL].unique()]
        symbols = [id_to_symbol.get(a, f"ID:{a}") for a in asset_ids]
        if len(symbols) < 2:
            continue

        ts_index = price_matrix.index
        day_start_pos = ts_index.searchsorted(day_start)
        day_end_pos = ts_index.searchsorted(day_end)
        window_start_pos = max(0, day_start_pos - warmup_candles)
        if day_start_pos - warmup_candles < 0:
            continue
        window_prices = price_matrix.iloc[window_start_pos:day_end_pos][asset_ids]
        window_prices.columns = symbols
        window_prices = window_prices.dropna(how="any")
        if len(window_prices) < warmup_candles + 10:
            continue

        spread_df_day = build_pairwise_return_spreads(window_prices)
        adjusted_df_day = compute_adjusted_spreads(spread_df_day, decay=eff["ewma_decay"])
        max_spread_df_day = compute_max_adjusted_spread(adjusted_df_day)
        tolerance_df_day = generate_tolerance_signals(
            max_spread_df_day, lookback_candles=lookback_candles,
            p_content=eff["p_content"], confidence=tb.FIXED_CONFIDENCE,
            recalib_every=eff["recalib_every"],
        )
        day_slice = adjusted_df_day.index[adjusted_df_day.index >= day_start]
        adjusted_df_day = adjusted_df_day.loc[day_slice]
        tolerance_df_day = tolerance_df_day.loc[day_slice]
        adjusted_df_day.columns = [tuple(sorted(c)) for c in adjusted_df_day.columns]

        price_spread_df_day, _hedge = build_pairwise_price_spreads(
            window_prices, hedge_lookback_candles=hedge_lookback_candles,
        )
        exit_mean_df_day = compute_exit_mean(price_spread_df_day, exit_lookback_candles=exit_lookback_candles)
        price_spread_df_day = price_spread_df_day.loc[day_slice]
        exit_mean_df_day = exit_mean_df_day.loc[day_slice]
        price_spread_df_day.columns = [tuple(sorted(c)) for c in price_spread_df_day.columns]
        exit_mean_df_day.columns = [tuple(sorted(c)) for c in exit_mean_df_day.columns]

        frames = []
        for pair_key in price_spread_df_day.columns:
            a, b = pair_key
            df = pd.DataFrame({
                "price_spread": price_spread_df_day[pair_key],
                "exit_mean": exit_mean_df_day[pair_key],
            })
            if pair_key in adjusted_df_day.columns:
                df["xt_ewma"] = adjusted_df_day[pair_key]
                df["upper_limit"] = tolerance_df_day["upper_limit"].values
            df["pair_a"] = a
            df["pair_b"] = b
            frames.append(df)
        if not frames:
            continue
        day_signals = pd.concat(frames, keys=range(len(frames)), names=["_pair_idx", "timestamp"])
        save_day_signals(variant="ewma", basket=basket, day=day, signals_df=day_signals)
        n_cached += 1

    print(f"EWMA/{basket}: {len(trades)} Trades, {n_cached}/{len(days)} Tage gecacht.")


_cache_ewma_result()

Erst Abschnitt 5b ausfuehren -- nichts zu cachen.


In [22]:
def _cache_bollinger_result():
    """Cached Abschnitt 6b/6c (Bollinger-Band-Variante)."""
    if not bo_last_result or "test_trades_df" not in bo_last_result:
        print("Erst Abschnitt 6b ausfuehren -- nichts zu cachen.")
        return

    trades = bo_last_result["test_trades_df"]
    result = bo_last_result["result"]
    best_params = bo_last_result["best_params"]
    basket = result.basket

    equity_curve = pte.compute_equity_curve(
        trades, initial_capital=tbo.FIXED_INITIAL_CAPITAL,
        max_concurrent_positions=best_params["max_concurrent_positions"],
    )
    stats = pte.compute_backtest_stats(trades, equity_curve)
    save_test_result(
        variant="bollinger", basket=basket,
        test_start=bo_last_result["test_start"], test_end=bo_last_result["test_end"],
        params=best_params, stats=stats, trades_df=trades, equity_curve=equity_curve,
    )

    if trades.empty:
        print(f"Bollinger/{basket}: keine Trades -- Cache nur mit leeren Tabellen geschrieben.")
        return

    price_matrix = ohlcv_df.pivot_table(
        index=TIMESTAMP_COL, columns=ASSET_ID_COL, values=CLOSE_COL, aggfunc="last",
    ).sort_index()
    cliques_df_day = bo_last_result["cliques_df"]
    lookback_candles = best_params["lookback_candles"]
    hedge_lookback_candles = best_params["hedge_lookback_candles"]
    exit_lookback_candles = best_params["exit_lookback_candles"]
    warmup_candles = max(lookback_candles, hedge_lookback_candles, exit_lookback_candles) + 30

    days = pd.DatetimeIndex(sorted(trades["discovery_date"].unique()))
    n_cached = 0
    for day in days:
        day_start = day.normalize()
        day_end = day_start + pd.Timedelta(days=1)
        clique_row = cliques_df_day.loc[cliques_df_day["discovery_date"] == day]
        if clique_row.empty:
            continue
        asset_ids = [a for a in clique_row.iloc[0]["asset_ids"] if a in ohlcv_df[ASSET_ID_COL].unique()]
        if len(asset_ids) < 2:
            continue

        ts_index = price_matrix.index
        day_start_pos = ts_index.searchsorted(day_start)
        day_end_pos = ts_index.searchsorted(day_end)
        window_start_pos = max(0, day_start_pos - warmup_candles)
        if day_start_pos - warmup_candles < 0:
            continue
        window_prices = price_matrix.iloc[window_start_pos:day_end_pos][asset_ids]
        window_prices = window_prices.dropna(how="any")
        if len(window_prices) < warmup_candles + 10:
            continue

        spread_df_day, _hedge = bpe.build_pairwise_price_spreads(
            window_prices, hedge_lookback_candles=hedge_lookback_candles,
        )
        bands_day = bpe.compute_bollinger_bands(
            spread_df_day, lookback_candles=lookback_candles, n_std=best_params["n_std"],
        )
        exit_mean_day = bpe.compute_exit_mean(spread_df_day, exit_lookback_candles=exit_lookback_candles)
        day_slice = spread_df_day.index[spread_df_day.index >= day_start]

        frames = []
        for pair, band in bands_day.items():
            a_sym = id_to_symbol.get(pair[0], f"ID:{pair[0]}")
            b_sym = id_to_symbol.get(pair[1], f"ID:{pair[1]}")
            df = band.loc[day_slice].copy()
            df["exit_mean"] = exit_mean_day[pair].loc[day_slice]
            df["pair_a"] = a_sym
            df["pair_b"] = b_sym
            frames.append(df)
        if not frames:
            continue
        day_signals = pd.concat(frames, keys=range(len(frames)), names=["_pair_idx", "timestamp"])
        save_day_signals(variant="bollinger", basket=basket, day=day, signals_df=day_signals)
        n_cached += 1

    print(f"Bollinger/{basket}: {len(trades)} Trades, {n_cached}/{len(days)} Tage gecacht.")


_cache_bollinger_result()

Erst Abschnitt 6b ausfuehren -- nichts zu cachen.


In [23]:
def _cache_garch_result():
    """Cached Abschnitt 7b/7c (GARCH-Toleranzintervall-Variante)."""
    if not gc_last_result or "test_trades_df" not in gc_last_result:
        print("Erst Abschnitt 7b ausfuehren -- nichts zu cachen.")
        return

    trades = gc_last_result["test_trades_df"]
    result = gc_last_result["result"]
    best_params = gc_last_result["best_params"]
    basket = result.basket

    equity_curve = pte.compute_equity_curve(
        trades, initial_capital=tgc.FIXED_INITIAL_CAPITAL,
        max_concurrent_positions=best_params["max_concurrent_positions"],
    )
    stats = pte.compute_backtest_stats(trades, equity_curve)
    save_test_result(
        variant="garch", basket=basket,
        test_start=gc_last_result["test_start"], test_end=gc_last_result["test_end"],
        params=best_params, stats=stats, trades_df=trades, equity_curve=equity_curve,
    )

    if trades.empty:
        print(f"GARCH/{basket}: keine Trades -- Cache nur mit leeren Tabellen geschrieben.")
        return

    price_matrix = ohlcv_df.pivot_table(
        index=TIMESTAMP_COL, columns=ASSET_ID_COL, values=CLOSE_COL, aggfunc="last",
    ).sort_index()
    cliques_df_day = gc_last_result["cliques_df"]
    lookback_candles = best_params["lookback_candles"]
    warmup_candles = lookback_candles + 30

    days = pd.DatetimeIndex(sorted(trades["discovery_date"].unique()))
    n_cached = 0
    print(f"GARCH/{basket}: caching {len(days)} Tage -- GARCH-Fit pro Paar/Tag, kann dauern.")
    for day in days:
        day_start = day.normalize()
        day_end = day_start + pd.Timedelta(days=1)
        clique_row = cliques_df_day.loc[cliques_df_day["discovery_date"] == day]
        if clique_row.empty:
            continue
        asset_ids = [a for a in clique_row.iloc[0]["asset_ids"] if a in ohlcv_df[ASSET_ID_COL].unique()]
        symbols = [id_to_symbol.get(a, f"ID:{a}") for a in asset_ids]
        if len(symbols) < 2:
            continue

        ts_index = price_matrix.index
        day_start_pos = ts_index.searchsorted(day_start)
        day_end_pos = ts_index.searchsorted(day_end)
        window_start_pos = max(0, day_start_pos - warmup_candles)
        if day_start_pos - warmup_candles < 0:
            continue
        window_prices = price_matrix.iloc[window_start_pos:day_end_pos][asset_ids]
        window_prices.columns = symbols
        window_prices = window_prices.dropna(how="any")
        if len(window_prices) < warmup_candles + 10:
            continue

        spread_df_day = build_pairwise_return_spreads(window_prices)
        adjusted_df_day, _fit_info = compute_garch_adjusted_spreads(spread_df_day)
        max_spread_df_day = compute_max_adjusted_spread(adjusted_df_day)
        tolerance_df_day = generate_tolerance_signals(
            max_spread_df_day, lookback_candles=lookback_candles,
            p_content=best_params["p_content"], confidence=tgc.FIXED_CONFIDENCE,
            recalib_every=best_params["recalib_every"],
        )
        day_slice = max_spread_df_day.index[max_spread_df_day.index >= day_start]
        max_spread_df_day = max_spread_df_day.loc[day_slice]
        tolerance_df_day = tolerance_df_day.loc[day_slice]

        day_signals = pd.DataFrame({
            "xmax": max_spread_df_day["xmax"],
            "upper_limit": tolerance_df_day["upper_limit"],
            "lower_limit": tolerance_df_day["lower_limit"],
            "exit_mid": (tolerance_df_day["lower_limit"] + tolerance_df_day["upper_limit"]) / 2.0,
        })
        save_day_signals(variant="garch", basket=basket, day=day, signals_df=day_signals)
        n_cached += 1

    print(f"GARCH/{basket}: {len(trades)} Trades, {n_cached}/{len(days)} Tage gecacht.")


_cache_garch_result()

Erst Abschnitt 7b ausfuehren -- nichts zu cachen.


## 8b. Ergebnis-Browser (rein aus Cache, kein OHLCV-/GARCH-Zugriff)

Dropdown waehlt eine gecachte Strategie/Basket-Kombination aus 8a, danach
Tag-Navigation wie in 5c/6c/7c -- alle Daten kommen aus
`data/processed/results_cache/`.

In [24]:
# =====================================================================
# Abschnitt 8b: Ergebnis-Browser -- rein aus dem Cache (Src.results_cache)
# =====================================================================
from Src.results_cache import list_cached_results, load_test_result, load_day_signals, cached_days

rb_cache_overview = list_cached_results()
if rb_cache_overview.empty:
    print("Noch kein Cache vorhanden -- erst Abschnitt 8a fuer mindestens eine Strategie/Basket-Kombination ausfuehren.")
else:
    display(rb_cache_overview)

rb_options = [
    (f"{row.variant} / {row.basket}", (row.variant, row.basket))
    for row in rb_cache_overview.itertuples()
] or [("(kein Cache)", None)]

rb_w_select = widgets.Dropdown(options=rb_options, description="Ergebnis:", style={"description_width": "80px"})
rb_w_day_label = widgets.HTML()
rb_w_prev_button = widgets.Button(description="< Tag", layout=widgets.Layout(width="90px"))
rb_w_next_button = widgets.Button(description="Tag >", layout=widgets.Layout(width="90px"))
rb_output = widgets.Output()
rb_state = {"day_idx": 0, "result": None}


def _rb_load_selected():
    choice = rb_w_select.value
    if choice is None:
        return None
    variant, basket = choice
    if rb_state["result"] is None or rb_state["result"]["variant"] != variant or rb_state["result"]["basket"] != basket:
        rb_state["result"] = load_test_result(variant=variant, basket=basket)
        rb_state["day_idx"] = 0
    return rb_state["result"]


def _rb_render(_=None):
    with rb_output:
        clear_output(wait=True)
        result = _rb_load_selected()
        if result is None:
            print("Kein Cache ausgewaehlt.")
            return

        variant, basket = result["variant"], result["basket"]
        trades = result["trades_df"]
        stats = result["stats"]

        print(f"{variant} / {basket} | Testzeitraum: {result['test_start']} -> {result['test_end']}")
        stats_display = pd.DataFrame([{
            "Anzahl Trades": stats.get("n_trades"),
            "Win-Rate": f"{stats['win_rate']:.1%}" if stats.get("win_rate") is not None else "n/a",
            "Sharpe": f"{stats['sharpe']:.2f}" if stats.get("sharpe") is not None else "n/a",
            "Max Drawdown": f"{stats['max_drawdown']:.2%}" if stats.get("max_drawdown") is not None else "n/a",
            "Gesamtrendite": f"{stats['total_return']:.2%}" if stats.get("total_return") is not None else "n/a",
            "Durchschnittliche Haltedauer": f"{stats['avg_holding_hours']:.1f}h" if stats.get("avg_holding_hours") is not None else "n/a",
        }]).T
        stats_display.columns = ["Wert"]
        display(stats_display)
        pte.plot_equity_curve(result["equity_curve"], initial_capital=result["params"].get("initial_capital", 10_000.0))

        days = cached_days(variant=variant, basket=basket)
        if not days:
            print("Keine Tages-Signalserien gecacht (Trades leer oder 8a noch nicht mit Tagesdaten gelaufen).")
            return
        rb_state["day_idx"] = max(0, min(rb_state["day_idx"], len(days) - 1))
        day = days[rb_state["day_idx"]]
        rb_w_day_label.value = f"<b>{day.date()}</b> &nbsp; (Tag {rb_state['day_idx'] + 1} von {len(days)})"

        day_signals = load_day_signals(variant=variant, basket=basket, day=day)
        day_trades = trades.loc[trades["discovery_date"] == pd.Timestamp(day)]

        import matplotlib.pyplot as plt
        from itertools import combinations

        if variant == "garch":
            fig, ax = plt.subplots(figsize=(11, 5))
            ax.plot(day_signals.index, day_signals["xmax"], color="#2a2a28", linewidth=1.1, label="Xmax")
            ax.plot(day_signals.index, day_signals["upper_limit"], color="#c1443f", linewidth=1, linestyle="--", label="Obere Toleranzgrenze")
            ax.plot(day_signals.index, day_signals["lower_limit"], color="#256abf", linewidth=1, linestyle="--", label="Untere Toleranzgrenze")
            ax.plot(day_signals.index, day_signals["exit_mid"], color="#1baf7a", linewidth=1.2, linestyle="-.", label="Exit-Mitte")
            for _, trade in day_trades.iterrows():
                if trade["entry_time"] in day_signals.index:
                    ax.scatter([trade["entry_time"]], [day_signals["xmax"].loc[trade["entry_time"]]], color="firebrick", zorder=5, s=45)
                if trade["exit_time"] in day_signals.index:
                    ax.scatter([trade["exit_time"]], [day_signals["xmax"].loc[trade["exit_time"]]], color="seagreen", zorder=5, s=45, marker="s")
            ax.legend(frameon=False, fontsize=7, ncol=2)
            ax.grid(True, alpha=0.25)
            ax.set_title(f"Xmax mit Wilks-Toleranzgrenzen und Exit-Mitte -- {basket} -- {day.date()}")
            plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
            plt.show()
        else:
            pair_groups = day_signals.groupby(["pair_a", "pair_b"])
            n_pairs = len(pair_groups)
            n_cols = min(2, n_pairs) or 1
            n_rows = int(np.ceil(n_pairs / n_cols)) or 1
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(8.5 * n_cols, 5.5 * n_rows), squeeze=False)

            for i, ((a, b), df) in enumerate(pair_groups):
                ax = axes[i // n_cols][i % n_cols]
                df = df.reset_index(level="_pair_idx", drop=True)
                if variant == "ewma":
                    ax.plot(df.index, df["price_spread"], color="steelblue", linewidth=1, label="Preis-Spread")
                    ax.plot(df.index, df["exit_mean"], color="#1baf7a", linewidth=1.2, linestyle="-.", label="Exit-Mittelwert")
                    if "xt_ewma" in df.columns:
                        ax2 = ax.twinx()
                        ax2.plot(df.index, df["xt_ewma"], linewidth=0.9, color="darkorange", alpha=0.8, label="Xt (EWMA)")
                        ax2.plot(df.index, df["upper_limit"], linewidth=0.8, color="firebrick", linestyle=":", alpha=0.7)
                        ax2.plot(df.index, -df["upper_limit"], linewidth=0.8, color="firebrick", linestyle=":", alpha=0.7)
                        ax2.tick_params(axis="y", labelsize=6, labelcolor="darkorange")
                else:  # bollinger
                    ax.plot(df.index, df["spread"], color="#2a2a28", linewidth=1.1, label="Preis-Spread")
                    ax.plot(df.index, df["mid"], color="#898781", linewidth=1, linestyle="--", label="Gleitender Mittelwert")
                    ax.plot(df.index, df["upper"], color="#c1443f", linewidth=0.8, linestyle=":", label="Oberes Band")
                    ax.plot(df.index, df["lower"], color="#256abf", linewidth=0.8, linestyle=":", label="Unteres Band")
                    ax.plot(df.index, df["exit_mean"], color="#1baf7a", linewidth=1.2, linestyle="-.", label="Exit-Mittelwert")

                pair_syms = {a, b}
                pair_trades = day_trades.loc[
                    day_trades.apply(lambda r: {r["asset_long_sym"], r["asset_short_sym"]} == pair_syms, axis=1)
                ]
                y_col = "price_spread" if variant == "ewma" else "spread"
                for _, trade in pair_trades.iterrows():
                    if trade["entry_time"] in df.index:
                        ax.scatter([trade["entry_time"]], [df[y_col].loc[trade["entry_time"]]], color="firebrick", zorder=5, s=35)
                    if trade["exit_time"] in df.index:
                        ax.scatter([trade["exit_time"]], [df[y_col].loc[trade["exit_time"]]], color="seagreen", zorder=5, s=35, marker="s")

                ax.set_title(f"{a}-{b}", fontsize=9)
                ax.tick_params(axis="x", rotation=30, labelsize=7)
                ax.tick_params(axis="y", labelsize=7)

            for j in range(n_pairs, n_rows * n_cols):
                axes[j // n_cols][j % n_cols].axis("off")
            fig.suptitle(f"Trades -- {basket} ({variant}) -- {day.date()}")
            plt.tight_layout()
            plt.show()

        display(day_trades[[
            "pair", "asset_long_sym", "asset_short_sym", "direction", "entry_time", "exit_time",
            "exit_reason", "gross_return", "net_return", "holding_candles",
        ]].reset_index(drop=True))


def _rb_on_select(_change=None):
    rb_state["day_idx"] = 0
    _rb_render()


def _rb_on_prev(_button=None):
    rb_state["day_idx"] -= 1
    _rb_render()


def _rb_on_next(_button=None):
    rb_state["day_idx"] += 1
    _rb_render()


rb_w_select.observe(_rb_on_select, names="value")
rb_w_prev_button.on_click(_rb_on_prev)
rb_w_next_button.on_click(_rb_on_next)

rb_ui = widgets.VBox([
    rb_w_select,
    widgets.HBox([rb_w_prev_button, rb_w_day_label, rb_w_next_button]),
    rb_output,
])
display(rb_ui)
_rb_render()

,variant,basket,test_start,test_end,n_trades,total_return,sharpe
0,garch,Basket 1,2025-02-08,2025-02-15,53,-0.012318,-132.150061
